<a href="https://colab.research.google.com/github/eduardoparente-rgb/Gerar-e-carregar-Imagens---Knowledge-Base/blob/main/Base_Dados_Lexica_WordNet_V7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Geração de imagens para os Conceitos — WordNet → ooinfo

Notebook independente (não depende dos outros notebooks do projeto rodarem antes, só dos arquivos que eles já geraram no Drive).

**O que este notebook faz**, para os **50 primeiros conceitos** de `conceitos.json`:

1. Usa uma LLM (**DeepSeek**, via `deepseek-v4-flash`) para transformar o termo preferencial + a descrição do conceito num **prompt de geração de imagem** bem escrito.
2. Manda esse prompt pro **Pollinations.ai** (gratuito, sem chave) pra gerar a imagem de fato.
3. Faz **upload** da imagem pro ooinfo e associa ao campo **arquivo** do item, via `PATCH` na API.

**Sobre custo:** o Pollinations.ai é gratuito e não pede chave. A API da DeepSeek **não tem um tier gratuito permanente** — mas toda conta nova ganha 5 milhões de tokens grátis (sem cartão) e os 50 prompts deste notebook usam uma fração mínima disso (algumas dezenas de milhares de tokens no total). Se a sua chave já gastou esse crédito, o custo residual é da ordem de centavos de dólar — mas não é tecnicamente R$ 0,00 nesse caso. Se você quiser garantia de custo zero absoluto, dá pra pular a célula 4 (DeepSeek) e usar o prompt simples da célula 4-alternativa no lugar — deixei as duas.

**Importante sobre o upload:** o formato exato que a API do ooinfo espera para associar um arquivo a um item **nunca foi testado neste projeto**. A seção 6 (diagnóstico) tenta os formatos mais prováveis com 1 item de teste antes de aplicar aos 50 conceitos de verdade — não pule essa etapa.

## 1. Instalação

Só a biblioteca `requests` é necessária (já vem no Colab); nenhuma biblioteca extra precisa ser instalada — nem para a DeepSeek nem para o Pollinations, já que os dois são chamados via API REST simples.

In [ ]:
import requests
import getpass
import json
import os
import time
import random
import urllib.parse
from datetime import datetime
from google.colab import drive

print("Pronto — nenhuma instalação extra necessária.")

Pronto — nenhuma instalação extra necessária.


## 2. Configurações gerais e localização dos arquivos no Drive

Monta o Drive e localiza automaticamente `conceitos.json` e `mapeamento_conceitos_ooinfo.json` (gerados nas etapas anteriores do projeto) dentro da pasta do grupo, em vez de usar um caminho fixo — mesmo padrão já usado nos outros notebooks, pra não quebrar se a estrutura de pastas mudar.

Também tenta localizar `conceitos_traduzidos.json` (da tradução em lote, que pode estar rodando agora): se existir e já tiver a tradução de algum desses 50 conceitos, ela é usada como contexto extra pro prompt de imagem. Se não existir ainda, o notebook segue normalmente usando só a definição em inglês — não é um requisito.

In [ ]:
BASE_URL = "https://ooinfo.org"
ID_LISTA_CONCEITOS = "cmsyyeh9d001b35btu2ginbbv"

drive.mount('/content/drive')
RAIZ_PROJETO = "/content/drive/MyDrive/Grupo 3 - RAGI 2026"
PASTA_WORDNET = os.path.join(RAIZ_PROJETO, "Datasets", "Wordnet")
PASTA_IMAGENS = os.path.join(PASTA_WORDNET, "imagens_conceitos")
os.makedirs(PASTA_IMAGENS, exist_ok=True)


def localizar_arquivo(nome_arquivo, raiz):
    for pasta_atual, _, arquivos in os.walk(raiz):
        if nome_arquivo in arquivos:
            return os.path.join(pasta_atual, nome_arquivo)
    return None


ARQUIVO_CONCEITOS = localizar_arquivo("conceitos.json", RAIZ_PROJETO)
ARQUIVO_MAPEAMENTO_CONCEITOS = localizar_arquivo("mapeamento_conceitos_ooinfo.json", RAIZ_PROJETO)
ARQUIVO_TRADUCOES = localizar_arquivo("conceitos_traduzidos.json", RAIZ_PROJETO)  # opcional

if not ARQUIVO_CONCEITOS:
    raise FileNotFoundError("Não encontrei conceitos.json no Drive.")
if not ARQUIVO_MAPEAMENTO_CONCEITOS:
    raise FileNotFoundError(
        "Não encontrei mapeamento_conceitos_ooinfo.json no Drive. "
        "Os conceitos precisam já ter sido importados pro ooinfo antes de rodar isto."
    )

print("conceitos.json:", ARQUIVO_CONCEITOS)
print("mapeamento_conceitos_ooinfo.json:", ARQUIVO_MAPEAMENTO_CONCEITOS)
print("conceitos_traduzidos.json:", ARQUIVO_TRADUCOES or "(ainda não encontrado — ok, é opcional)")

# Arquivo de checkpoint deste notebook (não interfere em nada da tradução)
ARQUIVO_PROGRESSO = os.path.join(PASTA_WORDNET, "progresso_imagens_conceitos.json")

with open(ARQUIVO_CONCEITOS, "r", encoding="utf-8") as f:
    conceitos = json.load(f)
with open(ARQUIVO_MAPEAMENTO_CONCEITOS, "r", encoding="utf-8") as f:
    mapeamento_conceitos = json.load(f)

traducoes_por_id = {}
if ARQUIVO_TRADUCOES:
    with open(ARQUIVO_TRADUCOES, "r", encoding="utf-8") as f:
        traducoes = json.load(f)
    traducoes_por_id = {t["id"]: t.get("definicao_pt") for t in traducoes if t.get("definicao_pt")}

print(f"\nConceitos carregados: {len(conceitos):,}")
print(f"Mapeamento (nosso id -> id interno ooinfo): {len(mapeamento_conceitos):,}")
print(f"Traduções PT-BR já disponíveis: {len(traducoes_por_id):,}")

Mounted at /content/drive
conceitos.json: /content/drive/MyDrive/Grupo 3 - RAGI 2026/Datasets/Wordnet/conceitos.json
mapeamento_conceitos_ooinfo.json: /content/drive/MyDrive/Grupo 3 - RAGI 2026/Datasets/Wordnet/mapeamento_conceitos_ooinfo.json
conceitos_traduzidos.json: /content/drive/MyDrive/Grupo 3 - RAGI 2026/Traducao/conceitos_traduzidos.json

Conceitos carregados: 43,895
Mapeamento (nosso id -> id interno ooinfo): 43,895
Traduções PT-BR já disponíveis: 39,782


## 3. Selecionar os 50 primeiros conceitos e montar os dados de cada um

Pega `conceitos[:50]` e, para cada um, resolve o **termo preferencial** (usando `palavras.json` pra transformar o ID da palavra em texto) e a **descrição** (a `definicao` em inglês do WordNet, sempre disponível; mais a tradução PT-BR quando já existir). Também descarta de imediato qualquer um dos 50 que ainda não tenha `id_interno` no mapeamento do ooinfo — isso pode acontecer se a importação de Conceitos ainda não tiver terminado.

In [ ]:
ARQUIVO_PALAVRAS = localizar_arquivo("palavras.json", RAIZ_PROJETO)
if not ARQUIVO_PALAVRAS:
    raise FileNotFoundError("Não encontrei palavras.json no Drive.")

with open(ARQUIVO_PALAVRAS, "r", encoding="utf-8") as f:
    palavras = json.load(f)
palavra_por_id = {p["id"]: p["palavra"] for p in palavras}

primeiros_50 = conceitos[:50]
itens_trabalho = []
sem_mapeamento = []

for c in primeiros_50:
    nosso_id = c["id"]
    id_interno = mapeamento_conceitos.get(nosso_id)
    if not id_interno:
        sem_mapeamento.append(nosso_id)
        continue

    termo_preferencial = palavra_por_id.get(c.get("termo_preferencial"), "?")
    descricao_en = (c.get("definicao") or "").strip()
    descricao_pt = traducoes_por_id.get(nosso_id)

    itens_trabalho.append({
        "id": nosso_id,
        "id_interno": id_interno,
        "termo_preferencial": termo_preferencial,
        "descricao_en": descricao_en,
        "descricao_pt": descricao_pt,
    })

print(f"Conceitos prontos para processar: {len(itens_trabalho)}/50")
if sem_mapeamento:
    print(f"\n⚠️ {len(sem_mapeamento)} dos 50 primeiros ainda não têm id_interno no ooinfo "
          f"(importação de Conceitos pode não ter terminado). IDs:")
    for i in sem_mapeamento:
        print("  -", i)

print("\nExemplo (primeiro item):")
print(json.dumps(itens_trabalho[0], ensure_ascii=False, indent=2))

Conceitos prontos para processar: 50/50

Exemplo (primeiro item):
{
  "id": "omw-pt-00001740-n",
  "id_interno": "cmt1k4jopusg035btslfb3dc0",
  "termo_preferencial": "ente",
  "descricao_en": "that which is perceived or known or inferred to have its own distinct existence (living or nonliving)",
  "descricao_pt": "aquilo que é percebido, conhecido ou inferido como tendo existência distinta própria (vivo ou não vivo)"
}


## 4. Gerar o prompt de imagem com a DeepSeek (LLM)

Manda o termo preferencial + a(s) descrição(ões) disponíveis pra `deepseek-v4-flash` (endpoint compatível com OpenAI, `POST /chat/completions`) e pede de volta um **prompt de imagem em inglês** — inglês porque modelos de geração de imagem (incluindo o que o Pollinations usa) tendem a responder melhor a prompts em inglês, independente do idioma da definição original. Tem retry com backoff pra erros transitórios (rate limit, 5xx), igual ao padrão usado no resto do projeto para o Gemini.

In [ ]:
MODELO_DEEPSEEK = "deepseek-v4-flash"
DEEPSEEK_URL = "https://api.deepseek.com/chat/completions"

MAX_TENTATIVAS_LLM = 4
BACKOFF_BASE = 3
BACKOFF_MAX = 30

SYSTEM_PROMPT_DEEPSEEK = (
    "Você transforma o termo e a definição de um conceito lexical num prompt de imagem "
    "em INGLÊS, pronto para um gerador de imagens de texto-para-imagem. Regras: "
    "(1) responda APENAS com o prompt, sem explicações, sem aspas, sem markdown; "
    "(2) descreva uma cena ou ilustração concreta que represente o CONCEITO, não a palavra "
    "escrita; (3) estilo consistente: ilustração digital simples, cores neutras, sem texto "
    "nem letras na imagem, sem marcas registradas nem personagens de terceiros; "
    "(4) no máximo 2-3 frases."
)


def montar_contexto_conceito(item):
    partes = [f"Termo: {item['termo_preferencial']}", f"Definição (inglês): {item['descricao_en']}"]
    if item.get("descricao_pt"):
        partes.append(f"Definição (português): {item['descricao_pt']}")
    return "\n".join(partes)


def gerar_prompt_imagem_deepseek(item, api_key):
    contexto = montar_contexto_conceito(item)
    ultimo_erro = None

    for tentativa in range(1, MAX_TENTATIVAS_LLM + 1):
        try:
            resp = requests.post(
                DEEPSEEK_URL,
                headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
                json={
                    "model": MODELO_DEEPSEEK,
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT_DEEPSEEK},
                        {"role": "user", "content": contexto},
                    ],
                    "temperature": 0.7,
                    "max_tokens": 150,
                },
                timeout=30,
            )
            if resp.status_code == 200:
                return resp.json()["choices"][0]["message"]["content"].strip()

            ultimo_erro = f"HTTP {resp.status_code}: {resp.text[:300]}"
            if resp.status_code not in (429, 500, 502, 503, 504):
                raise RuntimeError(ultimo_erro)

        except requests.exceptions.RequestException as e:
            ultimo_erro = str(e)

        if tentativa < MAX_TENTATIVAS_LLM:
            espera = min(BACKOFF_MAX, BACKOFF_BASE * (2 ** (tentativa - 1))) + random.uniform(0, 2)
            print(f"   ⚠️ DeepSeek falhou (tentativa {tentativa}/{MAX_TENTATIVAS_LLM}): {ultimo_erro}")
            print(f"   Aguardando {espera:.1f}s...")
            time.sleep(espera)

    raise RuntimeError(f"DeepSeek falhou após {MAX_TENTATIVAS_LLM} tentativas: {ultimo_erro}")


# --- Teste rápido com 1 conceito, antes de rodar tudo ---
deepseek_key = getpass.getpass("Cole sua API key da DeepSeek: ")

prompt_teste = gerar_prompt_imagem_deepseek(itens_trabalho[0], deepseek_key)
print("\nConceito:", itens_trabalho[0]["termo_preferencial"])
print("Prompt gerado pela DeepSeek:\n", prompt_teste)

KeyboardInterrupt: Interrupted by user

### 4-alternativa (opcional) — prompt simples, sem LLM, custo zero absoluto

Só existe pra quem preferir garantir R$ 0,00 sem depender de crédito da DeepSeek. Monta o prompt direto por template, sem chamar nenhuma API de texto. **Não é necessário rodar esta célula se você já rodou a célula 4 (DeepSeek)** — o resto do notebook usa a função `montar_prompt(...)`, que aponta pra uma das duas.

In [ ]:
def montar_prompt_template(item):
    descricao = item.get("descricao_pt") or item["descricao_en"]
    return (
        f"Simple conceptual digital illustration representing '{item['termo_preferencial']}': "
        f"{descricao}. Minimalist style, neutral background, no text or letters in the image."
    )

# Escolha qual gerador de prompt usar no restante do notebook:
#   - "deepseek" -> usa a célula 4 (gerar_prompt_imagem_deepseek)
#   - "template" -> usa esta célula (sem custo, sem chamada de API)
FONTE_DO_PROMPT = "deepseek"

def montar_prompt(item):
    if FONTE_DO_PROMPT == "deepseek":
        prompt = gerar_prompt_imagem_deepseek(item, deepseek_key)
        if prompt and prompt.strip():
            return prompt
        # DeepSeek devolveu vazio -- acontece em conceitos muito abstratos
        # (ex.: "ente", "entidade física", "abstração"). Cai pro template
        # em vez de gerar imagem a partir de um prompt em branco.
        print("   ⚠️ DeepSeek devolveu prompt vazio para este conceito -- usando template como fallback.")
        return montar_prompt_template(item)
    return montar_prompt_template(item)

## 5. Gerar a imagem com o Pollinations.ai

API pública e gratuita, sem chave nem cadastro (`image.pollinations.ai`). Recebe o prompt (gerado na seção 4) e devolve os bytes do PNG direto.

In [ ]:
def detectar_mime_imagem(conteudo_bytes):
    if conteudo_bytes[:3] == b"\xff\xd8\xff":
        return "image/jpeg"
    if conteudo_bytes[:8] == b"\x89PNG\r\n\x1a\n":
        return "image/png"
    return "application/octet-stream"  # não deveria acontecer, mas não trava o script


def gerar_imagem_pollinations(prompt, caminho_saida, largura=768, altura=768):
    """Retorna (caminho_saida, mime_tipo) -- o Pollinations pode devolver JPEG
    mesmo quando o nome do arquivo termina em .png, então o formato real dos
    bytes é detectado pela assinatura do arquivo, não pela extensão."""
    prompt_codificado = urllib.parse.quote(prompt)
    url = f"https://image.pollinations.ai/prompt/{prompt_codificado}"
    params = {"width": largura, "height": altura, "nologo": "true"}

    resp = requests.get(url, params=params, timeout=60)
    if resp.status_code != 200:
        raise RuntimeError(f"Pollinations falhou: {resp.status_code} - {resp.text[:200]}")

    with open(caminho_saida, "wb") as f:
        f.write(resp.content)

    mime_tipo = detectar_mime_imagem(resp.content)
    return caminho_saida, mime_tipo


# --- Teste rápido: gera a imagem do prompt de teste da célula 4 ---
caminho_teste = os.path.join(PASTA_IMAGENS, "_teste_pollinations.png")
caminho_teste, mime_teste = gerar_imagem_pollinations(prompt_teste, caminho_teste)
print("Imagem de teste salva em:", caminho_teste)
print("Formato real detectado:", mime_teste)
print("Abra o arquivo pelo painel de arquivos do Colab (ícone de pasta à esquerda) para conferir.")


## 6. Login no ooinfo e diagnóstico do upload de arquivo

**Não pule esta seção.** O formato que a API do ooinfo espera para associar um arquivo a um item nunca foi confirmado neste projeto. Duas partes:

- **6.1** — login e descoberta automática da chave interna do campo cujo rótulo contém "arquivo" (via `GET /api/lists/{listId}/fields`), pra não depender de descoberta manual.
- **6.2** — testa 4 formatos plausíveis de upload/associação num **único item de teste** (o primeiro dos 50) e imprime o status e a resposta de cada tentativa, pra você identificar qual funcionou antes de aplicar aos 50 de verdade.

In [ ]:
# --- 6.1 Login + descoberta automática do campo "arquivo" ---

email = input("Email do ooinfo: ")
senha = getpass.getpass("Senha do ooinfo: ")

login_resp = requests.post(f"{BASE_URL}/api/auth/login", json={"email": email, "password": senha}, timeout=30)
print("Status do login:", login_resp.status_code)
if login_resp.status_code not in (200, 201):
    raise RuntimeError(f"Falha no login: {login_resp.text}")

access_token = login_resp.json()["accessToken"]
headers = {"Authorization": f"Bearer {access_token}"}
print("Login realizado.")


def descobrir_campo_arquivo(token):
    resp = requests.get(f"{BASE_URL}/api/lists/{ID_LISTA_CONCEITOS}/fields", headers=headers, timeout=30)
    if resp.status_code != 200:
        raise RuntimeError(f"Não consegui listar os campos: {resp.status_code} - {resp.text[:300]}")

    campos = resp.json()

    print(f"Total de campos: {len(campos)}")
    for c in campos:
        print(f"  name={c.get('name')!r:30} key={c.get('key')!r:25} type={c.get('type')}")

    # Critério principal: o TIPO do campo, não o nome (o campo pode se chamar
    # "Imagem", "Anexo", "Capa", etc. -- nome nunca é confiável). Tipos comuns
    # para upload de arquivo/imagem em plataformas de lista: FILE, IMAGE, MEDIA, ATTACHMENT.
    tipos_arquivo = {"FILE", "IMAGE", "MEDIA", "ATTACHMENT", "UPLOAD"}
    candidatos = [c for c in campos if str(c.get("type", "")).upper() in tipos_arquivo]

    # Fallback: procura "arquivo" OU "imagem" no nome, caso o tipo venha diferente do esperado
    if not candidatos:
        candidatos = [
            c for c in campos
            if any(termo in str(c.get("name", "")).lower() for termo in ("arquivo", "imagem", "anexo", "foto", "capa"))
        ]

    if not candidatos:
        print("\n⚠️ Não encontrei automaticamente nenhum campo de arquivo/imagem.")
        print("Preencha CAMPO_ARQUIVO manualmente na célula abaixo, olhando a lista acima.")
        return None

    chave = candidatos[0].get("key") or candidatos[0].get("name")
    print(f"\n✅ Campo de arquivo encontrado automaticamente: nome={candidatos[0].get('name')!r} chave=\"{chave}\" type={candidatos[0].get('type')}")
    return chave


CAMPO_ARQUIVO = descobrir_campo_arquivo(access_token)
# Se a descoberta automática não funcionar, defina manualmente aqui, ex.:
# CAMPO_ARQUIVO = "novo_campo_10"

Email do ooinfo: eduardoparente@discente.ufg.br


AttributeError: 'function' object has no attribute 'getpass'

In [ ]:
# --- 6.2 DIAGNÓSTICO — testar formatos de upload com o item de teste ---

def diagnostico_upload_arquivo(token, item_id_interno, caminho_imagem, campo_arquivo, mime_tipo="image/png"):
    resultados = {}

    # Tentativa 1: multipart/form-data direto no PATCH do item
    with open(caminho_imagem, "rb") as f:
        resp1 = requests.patch(
            f"{BASE_URL}/api/lists/{ID_LISTA_CONCEITOS}/items/{item_id_interno}",
            headers={"Authorization": f"Bearer {token}"},
            files={"file": f},
            data={"field": campo_arquivo},
            timeout=60,
        )
    resultados["1_patch_multipart"] = (resp1.status_code, resp1.text[:300])

    # Tentativa 2: endpoint dedicado de upload -> referenciar a URL/id retornado no item
    with open(caminho_imagem, "rb") as f:
        resp2 = requests.post(
            f"{BASE_URL}/api/uploads",
            headers={"Authorization": f"Bearer {token}"},
            files={"file": f},
            timeout=60,
        )
    resultados["2_post_uploads_endpoint"] = (resp2.status_code, resp2.text[:300])

    # Tentativa 3: multipart num endpoint de arquivos vinculado ao item
    with open(caminho_imagem, "rb") as f:
        resp3 = requests.post(
            f"{BASE_URL}/api/lists/{ID_LISTA_CONCEITOS}/items/{item_id_interno}/files",
            headers={"Authorization": f"Bearer {token}"},
            files={"file": f},
            timeout=60,
        )
    resultados["3_post_item_files"] = (resp3.status_code, resp3.text[:300])

    # Tentativa 4: PATCH normal (JSON) com a imagem embutida como data URI em base64
    import base64
    with open(caminho_imagem, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("ascii")
    data_uri = f"data:{mime_tipo};base64,{b64}"
    resp4 = requests.patch(
        f"{BASE_URL}/api/lists/{ID_LISTA_CONCEITOS}/items/{item_id_interno}",
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
        json={"values": {campo_arquivo: data_uri}},
        timeout=60,
    )
    resultados["4_patch_json_base64"] = (resp4.status_code, resp4.text[:300])

    print("=" * 70)
    print("RESULTADOS DO DIAGNÓSTICO DE UPLOAD")
    print("=" * 70)
    for nome, (status, corpo) in resultados.items():
        marcador = "✅" if status in (200, 201) else "  "
        print(f"\n{marcador} [{nome}] status={status}")
        print(corpo)

    return resultados


item_teste = itens_trabalho[0]
resultados_diagnostico = diagnostico_upload_arquivo(
    access_token, item_teste["id_interno"], caminho_teste, CAMPO_ARQUIVO, mime_tipo=mime_teste
)

print("\n" + "=" * 70)
print("Confira acima qual tentativa deu status 200/201.")
print("Depois, defina FORMATO_UPLOAD_CONFIRMADO na próxima célula com o número (1, 2, 3 ou 4)")
print("correspondente, antes de rodar a seção 7 nos 50 conceitos de verdade.")
print("=" * 70)

### 6.3 Confirmar o formato que funcionou

Depois de olhar os resultados acima, defina qual das 4 tentativas retornou `200`/`201` — é essa que a seção 7 vai usar em escala.

In [ ]:
# Preencha com o número da tentativa que funcionou na CÉLULA 6.2 (1, 2, 3 ou 4).
# Se NENHUMA funcionou: abra o DevTools do navegador (aba Network), faça o upload manual
# de uma imagem em um item pelo site do ooinfo, e confira ali a requisição real — foi assim
# que outros formatos de API deste projeto foram descobertos (ex.: os slugs das listas).
# Confirmado em diagnóstico real: tentativa 4 (PATCH em JSON, imagem em base64
# dentro de values["imagem"]) -- as tentativas 2 e 3 deram 404 (endpoint não
# existe) e a 1 é um falso-positivo (não altera o campo "imagem" de verdade).
FORMATO_UPLOAD_CONFIRMADO = 4

if FORMATO_UPLOAD_CONFIRMADO not in (1, 2, 3, 4):
    print("⚠️ FORMATO_UPLOAD_CONFIRMADO ainda não foi definido. Edite a célula acima antes de continuar.")
else:
    print(f"Formato confirmado: tentativa {FORMATO_UPLOAD_CONFIRMADO}. Pronto para a seção 7.")

## 7. Aplicar aos 50 conceitos (produção, com checkpoint)

Para cada um dos 50 conceitos: gera o prompt (DeepSeek ou template, conforme `FONTE_DO_PROMPT` na seção 4-alternativa) → gera a imagem no Pollinations → faz o upload usando o formato confirmado na seção 6 → grava o progresso em `progresso_imagens_conceitos.json`. Tem checkpoint por item: se a execução parar no meio (erro, sessão do Colab caindo, etc.), rodar a célula de novo pula os que já foram concluídos com sucesso e continua dos que faltam.

In [ ]:
def salvar_json_seguro(caminho, dados):
    temporario = caminho + ".tmp"
    with open(temporario, "w", encoding="utf-8") as f:
        json.dump(dados, f, ensure_ascii=False, indent=2)
    os.replace(temporario, caminho)


def carregar_json(caminho, padrao):
    if not os.path.exists(caminho):
        return padrao
    try:
        with open(caminho, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print(f"⚠️ Erro lendo {caminho}: {e}")
        return padrao


def enviar_arquivo(token, item_id_interno, caminho_imagem, campo_arquivo, formato, mime_tipo="image/png"):
    if formato == 1:
        with open(caminho_imagem, "rb") as f:
            return requests.patch(
                f"{BASE_URL}/api/lists/{ID_LISTA_CONCEITOS}/items/{item_id_interno}",
                headers={"Authorization": f"Bearer {token}"},
                files={"file": f}, data={"field": campo_arquivo}, timeout=60,
            )
    if formato == 2:
        with open(caminho_imagem, "rb") as f:
            resp_upload = requests.post(
                f"{BASE_URL}/api/uploads",
                headers={"Authorization": f"Bearer {token}"},
                files={"file": f}, timeout=60,
            )
        if resp_upload.status_code not in (200, 201):
            return resp_upload
        referencia = resp_upload.json().get("url") or resp_upload.json().get("id")
        return requests.patch(
            f"{BASE_URL}/api/lists/{ID_LISTA_CONCEITOS}/items/{item_id_interno}",
            headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
            json={"values": {campo_arquivo: referencia}}, timeout=60,
        )
    if formato == 3:
        with open(caminho_imagem, "rb") as f:
            return requests.post(
                f"{BASE_URL}/api/lists/{ID_LISTA_CONCEITOS}/items/{item_id_interno}/files",
                headers={"Authorization": f"Bearer {token}"},
                files={"file": f}, timeout=60,
            )
    if formato == 4:
        import base64
        with open(caminho_imagem, "rb") as f:
            b64 = base64.b64encode(f.read()).decode("ascii")
        data_uri = f"data:{mime_tipo};base64,{b64}"
        return requests.patch(
            f"{BASE_URL}/api/lists/{ID_LISTA_CONCEITOS}/items/{item_id_interno}",
            headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
            json={"values": {campo_arquivo: data_uri}}, timeout=60,
        )
    raise ValueError(f"Formato de upload desconhecido: {formato}")


if FORMATO_UPLOAD_CONFIRMADO not in (1, 2, 3, 4):
    raise RuntimeError(
        "Defina FORMATO_UPLOAD_CONFIRMADO (seção 6.3) com o formato que funcionou "
        "no diagnóstico antes de rodar esta célula."
    )

progresso = carregar_json(ARQUIVO_PROGRESSO, {})  # nosso_id -> {"status": "ok"|"falha", ...}

concluidos = sum(1 for r in progresso.values() if r.get("status") == "ok")
print(f"Checkpoint: {concluidos}/{len(itens_trabalho)} já concluídos com sucesso anteriormente.")

inicio = time.time()

for i, item in enumerate(itens_trabalho, start=1):
    nosso_id = item["id"]

    if progresso.get(nosso_id, {}).get("status") == "ok":
        print(f"[{i}/{len(itens_trabalho)}] {nosso_id} — já concluído, pulando.")
        continue

    print(f"\n[{i}/{len(itens_trabalho)}] {nosso_id} ({item['termo_preferencial']})")

    try:
        prompt = montar_prompt(item)
        print("   Prompt:", prompt[:150] + ("..." if len(prompt) > 150 else ""))

        caminho_imagem = os.path.join(PASTA_IMAGENS, f"{nosso_id}.png")  # nome do arquivo; não define o formato real
        caminho_imagem, mime_imagem = gerar_imagem_pollinations(prompt, caminho_imagem)

        resp = enviar_arquivo(
            access_token, item["id_interno"], caminho_imagem, CAMPO_ARQUIVO, FORMATO_UPLOAD_CONFIRMADO,
            mime_tipo=mime_imagem,
        )

        if resp.status_code in (200, 201):
            progresso[nosso_id] = {"status": "ok", "prompt": prompt, "timestamp": datetime.now().isoformat()}
            print("   ✅ imagem gerada e associada.")
        else:
            progresso[nosso_id] = {
                "status": "falha", "erro": f"upload HTTP {resp.status_code}: {resp.text[:200]}",
                "prompt": prompt, "timestamp": datetime.now().isoformat(),
            }
            print(f"   ❌ falha no upload: HTTP {resp.status_code} — {resp.text[:200]}")

    except Exception as e:
        progresso[nosso_id] = {"status": "falha", "erro": str(e), "timestamp": datetime.now().isoformat()}
        print(f"   ❌ falha: {e}")

    salvar_json_seguro(ARQUIVO_PROGRESSO, progresso)
    time.sleep(1)  # respiro entre itens, gentil com DeepSeek/Pollinations/ooinfo

tempo_total = time.time() - inicio
sucessos = sum(1 for r in progresso.values() if r.get("status") == "ok")
falhas = sum(1 for r in progresso.values() if r.get("status") == "falha")

print("\n" + "=" * 70)
print("CONCLUÍDO")
print("=" * 70)
print(f"Sucesso: {sucessos}/{len(itens_trabalho)}")
print(f"Falhas:  {falhas}/{len(itens_trabalho)}")
print(f"Tempo desta execução: {tempo_total/60:.1f} min")
print(f"Progresso salvo em: {ARQUIVO_PROGRESSO}")

## 8. Relatório de falhas (opcional)

Só leitura — lê o checkpoint salvo na seção 7 e lista os conceitos que falharam, com o motivo, pra facilitar rodar de novo só o necessário (a célula 7 já pula os que deram certo automaticamente, então basta rodá-la de novo depois de investigar).

In [ ]:
progresso_final = carregar_json(ARQUIVO_PROGRESSO, {})
falhas_detalhadas = {k: v for k, v in progresso_final.items() if v.get("status") == "falha"}

print(f"Total de falhas registradas: {len(falhas_detalhadas)}")
for nosso_id, info in falhas_detalhadas.items():
    print(f"\n- {nosso_id}: {info.get('erro')}")

Testes

In [ ]:
# ============================================================
# GALERIA COMPARATIVA
# ============================================================

import html

resultados = carregar_json(
    ARQUIVO_RESULTADOS,
    []
)

por_conceito = {}

for r in resultados:

    if r.get("status") != "ok":
        continue

    por_conceito.setdefault(
        r["termo"],
        {}
    )[r["estrategia"]] = r


linhas = []

for item in AMOSTRA_TESTE:

    termo = item["termo"]

    dados = por_conceito.get(
        termo,
        {}
    )

    celulas = []

    for estrategia in ESTRATEGIAS:

        r = dados.get(
            estrategia
        )

        if r:

            imagem = os.path.basename(
                r["imagem"]
            )

            prompt = html.escape(
                r["prompt_visual"]
            )

            ideia = html.escape(
                r["ideia_central"]
            )

            celula = f"""
            <td>
                <h3>{estrategia}</h3>

                <img
                    src="{imagem}"
                    width="300"
                    style="border:1px solid #ccc;"
                >

                <p>
                    <b>Ideia:</b>
                    {ideia}
                </p>

                <details>
                    <summary>
                        Ver prompt
                    </summary>

                    <p>
                        {prompt}
                    </p>
                </details>
            </td>
            """

        else:

            celula = """
            <td>
                <h3>Não gerado</h3>
            </td>
            """

        celulas.append(
            celula
        )

    definicao = html.escape(
        item["en"]
    )

    linhas.append(
        f"""
        <tr>

            <td style="width:180px;">
                <h2>{html.escape(termo)}</h2>

                <b>Tipo:</b>
                {item["tipo_esperado"]}

                <p>
                    <b>Definição:</b><br>
                    {definicao}
                </p>
            </td>

            {''.join(celulas)}

        </tr>
        """
    )


html_final = f"""
<!DOCTYPE html>

<html>

<head>

<meta charset="UTF-8">

<title>
Teste de estratégias de geração de imagens
</title>

<style>

body {{
    font-family: Arial, sans-serif;
    margin: 20px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
}}

td, th {{
    border: 1px solid #ccc;
    padding: 12px;
    vertical-align: top;
}}

img {{
    max-width: 300px;
    height: auto;
}}

h1 {{
    margin-bottom: 5px;
}}

h2 {{
    margin-top: 0;
}}

h3 {{
    text-transform: uppercase;
}}

details {{
    margin-top: 10px;
}}

</style>

</head>

<body>

<h1>
Teste comparativo de estratégias
</h1>

<p>
A — Literal |
B — Cena concreta |
C — Source-Target-Meaning |
D — Pedagógica
</p>

<table>

<tr>

<th>
Conceito
</th>

<th>
Literal
</th>

<th>
Cena concreta
</th>

<th>
S-T-M
</th>

<th>
Pedagógica
</th>

</tr>

{''.join(linhas)}

</table>

</body>

</html>
"""


ARQUIVO_HTML = os.path.join(
    PASTA_TESTE,
    "galeria_comparativa.html"
)


with open(
    ARQUIVO_HTML,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        html_final
    )


print(
    "=" * 70
)

print(
    "GALERIA CRIADA"
)

print(
    ARQUIVO_HTML
)

print(
    "=" * 70
)

In [ ]:
# ============================================================
# TESTE V3 — ESTRATÉGIAS DE PROMPT SEM GEMINI
# ============================================================
#
# Objetivo:
#   Comparar diferentes estratégias de representação visual
#   para conceitos concretos e abstratos.
#
# NÃO usa Gemini.
# NÃO grava no OOInfo.
# NÃO altera nenhum dado da produção.
#
# Para cada conceito:
#   1. Literal
#   2. Cena concreta
#   3. Metáfora visual
#   4. Exemplo prototípico
#   5. Diagrama conceitual
#
# 12 conceitos x 5 estratégias = 60 imagens
#
# Todas as imagens são salvas no Google Drive.
# Existe checkpoint por imagem.
# ============================================================

import os
import json
import time
import random
import requests
import urllib.parse
import getpass
import html
from datetime import datetime

from google.colab import drive


# ============================================================
# 1. CONFIGURAÇÃO
# ============================================================

BASE_URL = "https://gen.pollinations.ai"

PASTA_BASE = "/content/drive/MyDrive/Grupo 3 - RAGI 2026"
PASTA_TESTE = os.path.join(
    PASTA_BASE,
    "Teste_Prompts_Imagem_V3"
)

PASTA_IMAGENS = os.path.join(
    PASTA_TESTE,
    "imagens"
)

os.makedirs(PASTA_IMAGENS, exist_ok=True)

ARQUIVO_RESULTADOS = os.path.join(
    PASTA_TESTE,
    "resultados_teste.json"
)

ARQUIVO_PROGRESSO = os.path.join(
    PASTA_TESTE,
    "progresso.json"
)

ARQUIVO_GALERIA = os.path.join(
    PASTA_TESTE,
    "galeria.html"
)


# ============================================================
# 2. AMOSTRA DE TESTE
# ============================================================

AMOSTRA = [

    # ---------------- CONCRETOS ----------------

    {
        "termo": "objeto",
        "en": "a tangible and visible entity; an entity that can cast a shadow",
        "pt": "uma entidade tangível e visível; uma entidade que pode projetar uma sombra",
        "tipo": "concreto"
    },

    {
        "termo": "criatura",
        "en": "a living thing that has (or can develop) the ability to act or function independently",
        "pt": "ser vivo que tem ou pode desenvolver a capacidade de agir ou funcionar independentemente",
        "tipo": "concreto"
    },

    {
        "termo": "bentos",
        "en": "organisms (plants and animals) that live at or near the bottom of a sea",
        "pt": "organismos (plantas e animais) que vivem no fundo do mar ou perto dele",
        "tipo": "concreto"
    },

    {
        "termo": "célula",
        "en": "(biology) the basic structural and functional unit of all organisms",
        "pt": "(biologia) a unidade estrutural e funcional básica de todos os organismos",
        "tipo": "concreto"
    },

    {
        "termo": "animal",
        "en": "a living organism characterized by voluntary movement",
        "pt": "organismo vivo caracterizado pelo movimento voluntário",
        "tipo": "concreto"
    },

    {
        "termo": "artefato",
        "en": "a man-made object taken as a whole",
        "pt": "um objeto fabricado pelo homem considerado como um todo",
        "tipo": "concreto"
    },

    # ---------------- ABSTRATOS ----------------

    {
        "termo": "abstração",
        "en": "a general concept formed by extracting common features from specific examples",
        "pt": "conceito geral formado pela extração de características comuns de exemplos específicos",
        "tipo": "abstrato"
    },

    {
        "termo": "condição",
        "en": "the way something is with respect to its main attributes",
        "pt": "o modo como algo se encontra em relação aos seus principais atributos",
        "tipo": "abstrato"
    },

    {
        "termo": "emoção",
        "en": "the experiencing of affective and emotional states",
        "pt": "a vivência de estados afetivos e emocionais",
        "tipo": "abstrato"
    },

    {
        "termo": "cognição",
        "en": "the psychological result of perception and learning and reasoning",
        "pt": "o resultado psicológico da percepção, do aprendizado e do raciocínio",
        "tipo": "abstrato"
    },

    {
        "termo": "magnitude",
        "en": "how much there is or how many there are of something that you can quantify",
        "pt": "a quantidade ou o número de algo que pode ser quantificado",
        "tipo": "abstrato"
    },

    {
        "termo": "relação social",
        "en": "a relation between living organisms (especially between people)",
        "pt": "uma relação entre organismos vivos, especialmente entre pessoas",
        "tipo": "abstrato"
    }
]


# ============================================================
# 3. ESTRATÉGIAS
# ============================================================

ESTRATEGIAS = {

    "01_literal": """
Represent the dictionary definition as literally and directly as possible.
Show the clearest physical manifestation of the concept.
Prefer a single main subject and only the contextual elements necessary
to communicate the definition.
Do not use text, labels, letters, words, symbols or captions.
""",

    "02_cena_concreta": """
Represent the meaning through one concrete real-world scene.
Instead of depicting the word itself, show people, animals, objects,
places or physical events that clearly demonstrate the definition.
The image should feel like a photograph or educational illustration
of an example of the concept.
Do not use text, labels, letters, words, symbols or captions.
""",

    "03_metafora_visual": """
Represent the meaning through a strong visual metaphor.
If the concept is abstract, DO NOT attempt to physically draw the
abstract concept itself.

Instead, choose one simple concrete situation whose visual relationship
clearly communicates the meaning.

The metaphor must be immediately understandable without text.
Avoid surrealism, random symbolism and overly artistic ambiguity.
Do not use text, labels, letters, words, symbols or captions.
""",

    "04_exemplo_prototipico": """
Represent the most recognizable and prototypical example of this concept.
Ask: what concrete scene would most people immediately associate with
the definition?

Use familiar objects, people, animals, actions or situations.
Prioritize semantic clarity over artistic creativity.

Do not use text, labels, letters, words, symbols or captions.
""",

    "05_diagrama_conceitual": """
Create a clean educational conceptual illustration.

Use a small number of concrete visual elements arranged spatially
so that their relationship communicates the definition.

The image should resemble a professional educational textbook illustration,
not an abstract piece of art.

Use visual relationships, grouping, scale, contrast, arrows only when
they are visually necessary, but NEVER use written labels or words.

No text, letters, captions or typography.
"""
}


# ============================================================
# 4. PROMPT BASE
# ============================================================

def construir_prompt(item, estrategia):

    estrategia_texto = ESTRATEGIAS[estrategia]

    termo = item["termo"]
    definicao = item["en"]
    tipo = item["tipo"]

    prompt = f"""
Create a clear educational illustration representing the following
dictionary concept.

CONCEPT:
{termo}

DICTIONARY DEFINITION:
{definicao}

CONCEPT TYPE:
{tipo}

VISUAL STRATEGY:
{estrategia_texto}

IMPORTANT SEMANTIC RULES:

1. The dictionary definition is the authoritative meaning.
2. The image must communicate the DEFINITION, not merely the appearance
   suggested by the word.
3. For abstract concepts, prefer concrete situations, relationships,
   objects or actions that communicate the meaning.
4. Avoid generic representations that could represent many unrelated
   concepts.
5. Avoid surrealism unless absolutely necessary.
6. Avoid decorative elements that do not contribute to meaning.
7. The main semantic idea must be visually obvious.
8. No text.
9. No letters.
10. No words.
11. No captions.
12. No watermark.
13. Clean educational illustration.
14. Simple composition.
15. Strong subject hierarchy.
16. Square composition.
"""

    return " ".join(prompt.split())


# ============================================================
# 5. CHECKPOINT
# ============================================================

def salvar_json_seguro(caminho, dados):

    temporario = caminho + ".tmp"

    with open(temporario, "w", encoding="utf-8") as f:
        json.dump(
            dados,
            f,
            ensure_ascii=False,
            indent=2
        )

    os.replace(temporario, caminho)


def carregar_json(caminho, padrao):

    if not os.path.exists(caminho):
        return padrao

    try:

        with open(caminho, "r", encoding="utf-8") as f:
            return json.load(f)

    except Exception as e:

        print(f"⚠️ Erro lendo checkpoint: {e}")

        return padrao


progresso = carregar_json(
    ARQUIVO_PROGRESSO,
    {}
)


# ============================================================
# 6. AUTENTICAÇÃO POLLINATIONS
# ============================================================

print("=" * 70)
print("AUTENTICAÇÃO POLLINATIONS")
print("=" * 70)

api_key = getpass.getpass(
    "Cole sua API key do Pollinations: "
).strip()

if not api_key:

    raise RuntimeError(
        "Nenhuma API key foi informada."
    )

print("✅ Chave recebida.")


# ============================================================
# 7. GERAÇÃO DE IMAGEM
# ============================================================

def gerar_imagem(prompt, caminho_saida, seed):

    prompt_encoded = urllib.parse.quote(
        prompt,
        safe=""
    )

    # API atual do Pollinations
    url = (
        f"{BASE_URL}/image/"
        f"{prompt_encoded}"
        f"?model=flux"
        f"&width=768"
        f"&height=768"
        f"&seed={seed}"
        f"&nologo=true"
    )

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Accept": "image/*"
    }

    ultima_resposta = None

    for tentativa in range(1, 5):

        try:

            print(
                f"   🎨 Gerando imagem "
                f"(tentativa {tentativa}/4)..."
            )

            resp = requests.get(
                url,
                headers=headers,
                timeout=180
            )

            ultima_resposta = resp

            if resp.status_code == 200:

                content_type = (
                    resp.headers.get(
                        "content-type",
                        ""
                    )
                )

                if not content_type.startswith("image"):

                    raise RuntimeError(
                        "Servidor respondeu 200, "
                        "mas não retornou imagem."
                    )

                with open(
                    caminho_saida,
                    "wb"
                ) as f:

                    f.write(resp.content)

                print(
                    f"   ✅ Imagem salva: "
                    f"{os.path.basename(caminho_saida)}"
                )

                return True

            if resp.status_code == 401:

                raise RuntimeError(
                    "HTTP 401 — API key rejeitada."
                )

            print(
                f"   ⚠️ HTTP {resp.status_code}: "
                f"{resp.text[:300]}"
            )

        except Exception as e:

            print(
                f"   ⚠️ Erro: {str(e)[:300]}"
            )

        if tentativa < 4:

            espera = tentativa * 5

            print(
                f"   aguardando {espera}s..."
            )

            time.sleep(espera)

    if ultima_resposta is not None:

        raise RuntimeError(
            f"Falha Pollinations: "
            f"HTTP {ultima_resposta.status_code}: "
            f"{ultima_resposta.text[:500]}"
        )

    raise RuntimeError(
        "Falha de conexão com Pollinations."
    )


# ============================================================
# 8. TESTE REAL DA API
# ============================================================

print()
print("=" * 70)
print("TESTE DE CONEXÃO")
print("=" * 70)

prompt_teste = """
A simple red apple on a white table,
clean educational illustration,
no text, square composition
"""

arquivo_teste = os.path.join(
    PASTA_TESTE,
    "_teste_conexao.png"
)

try:

    gerar_imagem(
        prompt_teste,
        arquivo_teste,
        seed=12345
    )

    print()
    print("✅ TESTE DA API PASSOU.")
    print(
        "A imagem de teste foi salva em:"
    )
    print(arquivo_teste)

except Exception as e:

    print()
    print("❌ TESTE DA API FALHOU.")
    print(str(e))

    raise RuntimeError(
        "\n"
        "A API do Pollinations não respondeu corretamente.\n"
        "Nenhuma imagem do experimento será processada.\n"
        "Corrija a autenticação antes de continuar."
    )


# ============================================================
# 9. MONTAR TRABALHO
# ============================================================

trabalhos = []

for item in AMOSTRA:

    termo_slug = (
        item["termo"]
        .lower()
        .replace(" ", "_")
        .replace("ã", "a")
        .replace("á", "a")
        .replace("é", "e")
        .replace("ê", "e")
        .replace("í", "i")
        .replace("ó", "o")
        .replace("ô", "o")
        .replace("ú", "u")
        .replace("ç", "c")
    )

    for estrategia in ESTRATEGIAS:

        chave = (
            f"{termo_slug}__{estrategia}"
        )

        trabalhos.append({
            "chave": chave,
            "termo": item["termo"],
            "en": item["en"],
            "pt": item["pt"],
            "tipo": item["tipo"],
            "estrategia": estrategia,
            "prompt": construir_prompt(
                item,
                estrategia
            )
        })


# ============================================================
# 10. CHECKPOINT
# ============================================================

concluidos = sum(
    1
    for chave, registro in progresso.items()
    if registro.get("status") == "ok"
)

print()
print("=" * 70)
print("CHECKPOINT")
print("=" * 70)

print(
    f"Imagens já concluídas: "
    f"{concluidos}/{len(trabalhos)}"
)

print(
    f"Faltam: "
    f"{len(trabalhos) - concluidos}"
)


# ============================================================
# 11. PROCESSAMENTO
# ============================================================

resultados = []

inicio = time.time()

for indice, trabalho in enumerate(
    trabalhos,
    start=1
):

    chave = trabalho["chave"]

    if (
        progresso.get(chave, {})
        .get("status") == "ok"
    ):

        print(
            f"[{indice}/{len(trabalhos)}] "
            f"{chave} — já concluído."
        )

        resultados.append(
            progresso[chave]
        )

        continue

    print()
    print("-" * 70)

    print(
        f"[{indice}/{len(trabalhos)}] "
        f"{trabalho['termo']} — "
        f"{trabalho['estrategia']}"
    )

    nome_arquivo = (
        f"{chave}.png"
    )

    caminho = os.path.join(
        PASTA_IMAGENS,
        nome_arquivo
    )

    seed = (
        abs(hash(chave)) % 2147483647
    )

    try:

        gerar_imagem(
            trabalho["prompt"],
            caminho,
            seed
        )

        registro = {

            "chave": chave,

            "status": "ok",

            "termo": trabalho["termo"],

            "tipo": trabalho["tipo"],

            "estrategia": trabalho["estrategia"],

            "en": trabalho["en"],

            "pt": trabalho["pt"],

            "prompt": trabalho["prompt"],

            "imagem": caminho,

            "seed": seed,

            "timestamp":
                datetime.now().isoformat()
        }

        progresso[chave] = registro

        resultados.append(registro)

        print("   ✅ SUCESSO")

    except Exception as e:

        registro = {

            "chave": chave,

            "status": "falha",

            "termo": trabalho["termo"],

            "tipo": trabalho["tipo"],

            "estrategia": trabalho["estrategia"],

            "en": trabalho["en"],

            "pt": trabalho["pt"],

            "prompt": trabalho["prompt"],

            "erro": str(e),

            "timestamp":
                datetime.now().isoformat()
        }

        progresso[chave] = registro

        resultados.append(registro)

        print(
            f"   ❌ FALHA: {e}"
        )

    # salva após CADA imagem
    salvar_json_seguro(
        ARQUIVO_PROGRESSO,
        progresso
    )

    time.sleep(2)


# ============================================================
# 12. RESULTADOS
# ============================================================

salvar_json_seguro(
    ARQUIVO_PROGRESSO,
    progresso
)

salvar_json_seguro(
    ARQUIVO_RESULTADOS,
    list(progresso.values())
)


# ============================================================
# 13. GALERIA HTML
# ============================================================

linhas = []

for item in AMOSTRA:

    termo = item["termo"]

    slug = (
        termo
        .lower()
        .replace(" ", "_")
        .replace("ã", "a")
        .replace("á", "a")
        .replace("é", "e")
        .replace("ê", "e")
        .replace("í", "i")
        .replace("ó", "o")
        .replace("ô", "o")
        .replace("ú", "u")
        .replace("ç", "c")
    )

    celulas = []

    for estrategia in ESTRATEGIAS:

        chave = (
            f"{slug}__{estrategia}"
        )

        r = progresso.get(chave)

        if r and r.get("status") == "ok":

            caminho_relativo = os.path.relpath(
                r["imagem"],
                PASTA_TESTE
            )

            celulas.append(
                f"""
                <td>
                    <h3>{estrategia}</h3>
                    <img
                        src="{html.escape(caminho_relativo)}"
                        width="300"
                    >
                    <p>
                        <b>Prompt:</b><br>
                        {html.escape(r["prompt"])}
                    </p>
                </td>
                """

            )

        else:

            erro = (
                r.get("erro", "não gerada")
                if r else
                "não processada"
            )

            celulas.append(
                f"""
                <td>
                    <h3>{estrategia}</h3>
                    <p>❌ {html.escape(erro)}</p>
                </td>
                """
            )

    linhas.append(
        f"""
        <tr>
            <td>
                <h2>{html.escape(termo)}</h2>

                <b>Tipo:</b>
                {html.escape(item["tipo"])}

                <p>
                <b>Definição:</b><br>
                {html.escape(item["pt"])}
                </p>
            </td>

            {''.join(celulas)}
        </tr>
        """
    )


html_final = f"""
<!DOCTYPE html>

<html lang="pt-BR">

<head>

<meta charset="UTF-8">

<title>
Teste de estratégias de geração de imagens
</title>

<style>

body {{
    font-family: Arial, sans-serif;
    margin: 30px;
}}

table {{
    border-collapse: collapse;
    width: 100%;
}}

td {{
    border: 1px solid #ccc;
    padding: 15px;
    vertical-align: top;
}}

img {{
    max-width: 300px;
    height: auto;
}}

h1 {{
    margin-bottom: 30px;
}}

.prompt {{
    font-size: 12px;
}}

</style>

</head>

<body>

<h1>
Teste de estratégias de geração de imagens
</h1>

<p>
Comparação entre cinco estratégias para representar
conceitos concretos e abstratos.
</p>

<table>

<tr>

<th>
Conceito
</th>

<th>
Literal
</th>

<th>
Cena concreta
</th>

<th>
Metáfora visual
</th>

<th>
Exemplo prototípico
</th>

<th>
Diagrama conceitual
</th>

</tr>

{''.join(linhas)}

</table>

</body>

</html>
"""


with open(
    ARQUIVO_GALERIA,
    "w",
    encoding="utf-8"
) as f:

    f.write(html_final)


# ============================================================
# 14. RELATÓRIO
# ============================================================

sucessos = sum(
    1
    for r in progresso.values()
    if r.get("status") == "ok"
)

falhas = sum(
    1
    for r in progresso.values()
    if r.get("status") == "falha"
)

tempo = time.time() - inicio

print()
print("=" * 70)

print("TESTE FINALIZADO")

print("=" * 70)

print(
    f"Total de imagens: "
    f"{len(trabalhos)}"
)

print(
    f"Sucessos: "
    f"{sucessos}"
)

print(
    f"Falhas: "
    f"{falhas}"
)

print(
    f"Tempo: "
    f"{tempo/60:.1f} minutos"
)

print()
print(
    "📁 Pasta:"
)

print(
    PASTA_TESTE
)

print()
print(
    "🌐 Galeria:"
)

print(
    ARQUIVO_GALERIA
)

print()
print(
    "📄 Resultados:"
)

print(
    ARQUIVO_RESULTADOS
)

AUTENTICAÇÃO POLLINATIONS
Cole sua API key do Pollinations: ··········
✅ Chave recebida.

TESTE DE CONEXÃO
   🎨 Gerando imagem (tentativa 1/4)...
   ⚠️ Erro: HTTP 401 — API key rejeitada.
   aguardando 5s...
   🎨 Gerando imagem (tentativa 2/4)...
   ⚠️ Erro: HTTP 401 — API key rejeitada.
   aguardando 10s...
   🎨 Gerando imagem (tentativa 3/4)...
   ⚠️ Erro: HTTP 401 — API key rejeitada.
   aguardando 15s...
   🎨 Gerando imagem (tentativa 4/4)...
   ⚠️ Erro: HTTP 401 — API key rejeitada.

❌ TESTE DA API FALHOU.
Falha Pollinations: HTTP 401: {"success":false,"error":{"message":"A valid API key is required. Get one at https://enter.pollinations.ai/keys","code":"UNAUTHORIZED","timestamp":"2026-08-28T18:45:00.672Z"},"status":401}


RuntimeError: 
A API do Pollinations não respondeu corretamente.
Nenhuma imagem do experimento será processada.
Corrija a autenticação antes de continuar.

In [ ]:
# ================================================================
# TESTE CONTROLADO DE ESTRATÉGIAS DE PROMPT
# WORDNET / OOINFO / POLLINATIONS
#
# NÃO USA GEMINI
# NÃO USA API KEY
# NÃO ALTERA OOINFO
#
# 10 conceitos × 5 estratégias = 50 imagens
# ================================================================

import os
import json
import time
import urllib.parse
import requests
from datetime import datetime
from google.colab import drive


# ================================================================
# 1. DRIVE
# ================================================================

drive.mount("/content/drive")

PASTA_TESTE = (
    "/content/drive/MyDrive/"
    "Grupo 3 - RAGI 2026/"
    "Teste_Prompts_Imagem_V3"
)

os.makedirs(PASTA_TESTE, exist_ok=True)

ARQUIVO_RESULTADOS = os.path.join(
    PASTA_TESTE,
    "resultados_teste.json"
)


# ================================================================
# 2. CONCEITOS
# ================================================================

CONCEITOS = [

    # ------------------------------------------------------------
    # CONCRETOS
    # ------------------------------------------------------------

    {
        "termo": "objeto",
        "tipo": "concreto",
        "definicao": (
            "a tangible and visible entity; "
            "an entity that can cast a shadow"
        )
    },

    {
        "termo": "célula",
        "tipo": "concreto",
        "definicao": (
            "the basic structural and functional unit "
            "of all organisms"
        )
    },

    {
        "termo": "animal",
        "tipo": "concreto",
        "definicao": (
            "a living organism characterized by "
            "voluntary movement"
        )
    },

    {
        "termo": "artefato",
        "tipo": "concreto",
        "definicao": (
            "a man-made object taken as a whole"
        )
    },

    {
        "termo": "bentos",
        "tipo": "concreto",
        "definicao": (
            "organisms (plants and animals) that live "
            "at or near the bottom of a sea"
        )
    },

    # ------------------------------------------------------------
    # ABSTRATOS
    # ------------------------------------------------------------

    {
        "termo": "abstração",
        "tipo": "abstrato",
        "definicao": (
            "a general concept formed by extracting "
            "common features from specific examples"
        )
    },

    {
        "termo": "condição",
        "tipo": "abstrato",
        "definicao": (
            "the way something is with respect to "
            "its main attributes"
        )
    },

    {
        "termo": "emoção",
        "tipo": "abstrato",
        "definicao": (
            "the experiencing of affective and "
            "emotional states"
        )
    },

    {
        "termo": "cognição",
        "tipo": "abstrato",
        "definicao": (
            "the psychological result of perception "
            "and learning and reasoning"
        )
    },

    {
        "termo": "magnitude",
        "tipo": "abstrato",
        "definicao": (
            "how much there is or how many there are "
            "of something that you can quantify"
        )
    }
]


# ================================================================
# 3. ESTRATÉGIAS
# ================================================================

ESTRATEGIAS = {

    "01_literal": """
Represent the dictionary definition as literally as possible.
Use the most direct visual representation of the concept.
Show the concept itself or its most obvious physical manifestation.
Avoid symbolic or artistic interpretation.
Create a clear educational illustration.
""",

    "02_exemplo": """
Do not try to visually draw the abstract word itself.
Instead, show one concrete real-world example or situation
that clearly demonstrates the meaning of the dictionary definition.
The viewer should understand the concept from the situation.
Use an educational illustration style.
""",

    "03_metafora": """
Represent the meaning using a concrete visual metaphor.
Do not write the abstract concept.
Do not show abstract glowing shapes or generic conceptual graphics.
Choose a simple physical scene whose relationships, actions,
or visual structure communicate the meaning of the definition.
The metaphor must be understandable without text.
""",

    "04_cena_didatica": """
Create a clear educational scene that explains the dictionary
definition visually.
Use concrete objects, people, organisms, environments,
actions, spatial relationships, or comparisons as appropriate.
The image should function like an encyclopedia illustration:
simple, concrete, informative, and immediately understandable.
""",

    "05_representacao_semantica": """
Focus on the semantic meaning rather than the word itself.
Identify the essential entities, relationships, actions,
properties, or situations contained in the dictionary definition.
Represent those elements together in one coherent concrete scene.
Do not use text, labels, letters, diagrams, abstract glowing effects,
or generic AI symbolism.
"""
}


# ================================================================
# 4. CONSTRUÇÃO DOS PROMPTS
# ================================================================

def montar_prompt(conceito, estrategia):

    instrucao = ESTRATEGIAS[estrategia]

    prompt = f"""
Create a clean educational illustration representing the following
dictionary concept.

Concept type: {conceito["tipo"]}

Dictionary definition:
"{conceito["definicao"]}"

VISUAL STRATEGY:
{instrucao}

IMPORTANT RULES:

- The image must communicate the meaning of the definition.
- Use concrete visual elements whenever possible.
- No written words.
- No labels.
- No letters.
- No captions.
- No typography.
- No abstract glowing symbols.
- No surreal floating concepts.
- No generic "AI concept" imagery.
- Do not simply write or visualize the concept name.
- The result should look like a clear educational encyclopedia illustration.
- Use a simple composition with a clear visual subject.
- Square composition.
- Clean background.
- High visual clarity.
- One coherent scene.
"""

    return " ".join(prompt.split())


# ================================================================
# 5. GERADOR POLLINATIONS
# ================================================================

def gerar_imagem(prompt, caminho, seed):

    encoded = urllib.parse.quote(prompt, safe="")

    url = (
        "https://image.pollinations.ai/prompt/"
        f"{encoded}"
    )

    params = {
        "model": "flux",
        "width": 768,
        "height": 768,
        "seed": seed,
        "nologo": "true"
    }

    resposta = requests.get(
        url,
        params=params,
        timeout=180
    )

    resposta.raise_for_status()

    content_type = resposta.headers.get(
        "content-type",
        ""
    ).lower()

    if "image" not in content_type:
        raise RuntimeError(
            f"Resposta não é imagem: "
            f"{content_type}\n"
            f"{resposta.text[:500]}"
        )

    with open(caminho, "wb") as f:
        f.write(resposta.content)

    return content_type


# ================================================================
# 6. CHECKPOINT
# ================================================================

if os.path.exists(ARQUIVO_RESULTADOS):

    try:

        with open(
            ARQUIVO_RESULTADOS,
            "r",
            encoding="utf-8"
        ) as f:
            resultados = json.load(f)

    except Exception:

        print("⚠️ JSON existente inválido. Iniciando novo.")
        resultados = {}

else:

    resultados = {}


def salvar_checkpoint():

    temporario = ARQUIVO_RESULTADOS + ".tmp"

    with open(
        temporario,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            resultados,
            f,
            ensure_ascii=False,
            indent=2
        )

    os.replace(
        temporario,
        ARQUIVO_RESULTADOS
    )


# ================================================================
# 7. CONTAGEM
# ================================================================

TOTAL = len(CONCEITOS) * len(ESTRATEGIAS)

concluidos = sum(
    1
    for r in resultados.values()
    if r.get("status") == "ok"
)

print("=" * 70)
print("TESTE CONTROLADO DE ESTRATÉGIAS")
print("=" * 70)

print(f"Conceitos: {len(CONCEITOS)}")
print(f"Estratégias: {len(ESTRATEGIAS)}")
print(f"Total de imagens: {TOTAL}")
print(f"Já concluídas: {concluidos}")
print(f"Faltam: {TOTAL - concluidos}")

print("\nPasta:")
print(PASTA_TESTE)


# ================================================================
# 8. GERAÇÃO
# ================================================================

inicio = time.time()

contador = 0

for conceito in CONCEITOS:

    termo = conceito["termo"]

    slug = (
        termo.lower()
        .replace(" ", "_")
        .replace("ã", "a")
        .replace("á", "a")
        .replace("é", "e")
        .replace("ê", "e")
        .replace("í", "i")
        .replace("ó", "o")
        .replace("ô", "o")
        .replace("ú", "u")
        .replace("ç", "c")
    )

    # Mesma seed para todas as estratégias deste conceito.
    # Isso ajuda a reduzir a variação aleatória.
    seed = 1000 + CONCEITOS.index(conceito)

    print("\n")
    print("=" * 70)
    print(f"CONCEITO: {termo}")
    print(f"Tipo: {conceito['tipo']}")
    print("=" * 70)

    for estrategia in ESTRATEGIAS:

        contador += 1

        chave = f"{slug}__{estrategia}"

        if resultados.get(chave, {}).get("status") == "ok":

            print(
                f"[{contador}/{TOTAL}] "
                f"{termo} — {estrategia} "
                f"→ já concluído"
            )

            continue

        print(
            f"\n[{contador}/{TOTAL}] "
            f"{termo} — {estrategia}"
        )

        prompt = montar_prompt(
            conceito,
            estrategia
        )

        print("Prompt criado.")

        nome_imagem = (
            f"{slug}__{estrategia}.jpg"
        )

        caminho = os.path.join(
            PASTA_TESTE,
            nome_imagem
        )

        try:

            print("🎨 Gerando imagem...")

            content_type = gerar_imagem(
                prompt,
                caminho,
                seed
            )

            resultados[chave] = {

                "status": "ok",

                "termo": termo,

                "tipo": conceito["tipo"],

                "definicao": conceito["definicao"],

                "estrategia": estrategia,

                "prompt": prompt,

                "seed": seed,

                "arquivo": caminho,

                "content_type": content_type,

                "timestamp": datetime.now().isoformat()
            }

            print("✅ Imagem gerada.")

        except Exception as e:

            resultados[chave] = {

                "status": "falha",

                "termo": termo,

                "tipo": conceito["tipo"],

                "definicao": conceito["definicao"],

                "estrategia": estrategia,

                "prompt": prompt,

                "seed": seed,

                "erro": str(e),

                "timestamp": datetime.now().isoformat()
            }

            print("❌ Falha:")
            print(str(e))

        salvar_checkpoint()

        # Respeita o intervalo da API antiga.
        time.sleep(6)


# ================================================================
# 9. RESUMO
# ================================================================

tempo = time.time() - inicio

sucessos = sum(
    1
    for r in resultados.values()
    if r.get("status") == "ok"
)

falhas = sum(
    1
    for r in resultados.values()
    if r.get("status") == "falha"
)

print("\n")
print("=" * 70)
print("TESTE FINALIZADO")
print("=" * 70)

print(f"Sucessos: {sucessos}/{TOTAL}")
print(f"Falhas: {falhas}/{TOTAL}")
print(f"Tempo: {tempo / 60:.1f} minutos")

print("\nResultados:")
print(ARQUIVO_RESULTADOS)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TESTE CONTROLADO DE ESTRATÉGIAS
Conceitos: 10
Estratégias: 5
Total de imagens: 50
Já concluídas: 0
Faltam: 50

Pasta:
/content/drive/MyDrive/Grupo 3 - RAGI 2026/Teste_Prompts_Imagem_V3


CONCEITO: objeto
Tipo: concreto

[1/50] objeto — 01_literal
Prompt criado.
🎨 Gerando imagem...
✅ Imagem gerada.

[2/50] objeto — 02_exemplo
Prompt criado.
🎨 Gerando imagem...
✅ Imagem gerada.

[3/50] objeto — 03_metafora
Prompt criado.
🎨 Gerando imagem...
✅ Imagem gerada.

[4/50] objeto — 04_cena_didatica
Prompt criado.
🎨 Gerando imagem...
✅ Imagem gerada.

[5/50] objeto — 05_representacao_semantica
Prompt criado.
🎨 Gerando imagem...
✅ Imagem gerada.


CONCEITO: célula
Tipo: concreto

[6/50] célula — 01_literal
Prompt criado.
🎨 Gerando imagem...
✅ Imagem gerada.

[7/50] célula — 02_exemplo
Prompt criado.
🎨 Gerando imagem...
✅ Imagem gerada.

[8/50] célula — 03_metafora
Prompt

In [ ]:
"""
Teste comparativo de PROVEDORES de geração de imagem (Pollinations vs.
Hugging Face vs. Replicate), usando o MESMO prompt pra cada conceito —
gerado 1 vez pela estratégia adaptativa (classifica concreto/abstrato e
usa metáfora Source-Target-Meaning quando abstrato).

Amostra balanceada: 6 conceitos CONCRETOS + 6 ABSTRATOS (não usa os
hiperônimos do topo da hierarquia, que são o pior caso possível pra imagem).

Gera uma galeria HTML com 1 linha por conceito e 1 coluna por provedor,
pra comparação visual lado a lado. NÃO grava nada no ooinfo — é só teste.

Antes de rodar, numa célula separada:
  !pip install -q google-genai replicate huggingface_hub

Chaves necessárias (todas grátis de CRIAR — o uso de algumas tem custo):
  - Google AI Studio: aistudio.google.com (já usamos antes; obrigatória)
  - Pollinations: enter.pollinations.ai (a Isabelle já tem; opcional)
  - Hugging Face: huggingface.co/settings/tokens (token "read"; opcional)
  - Replicate: replicate.com/account/api-tokens (pede cartão pra ativar
    billing, mas o custo deste teste é centavos de dólar; opcional)

Deixe em branco qualquer uma das 3 chaves de imagem pra pular esse
provedor — não precisa ter as 3 pra rodar o teste.
"""

import json
import os
import time
import getpass
import requests
import urllib.parse
from google.colab import drive
from google import genai
from pydantic import BaseModel, Field

MODELO_TEXTO = "gemini-3.5-flash-lite"

# --- Amostra balanceada, tirada dos conceitos que a Isabelle já traduziu ---
AMOSTRA_TESTE = [
    # concretos
    {"termo": "objeto", "en": "a tangible and visible entity; an entity that can cast a shadow",
     "pt": "uma entidade tangível e visível; uma entidade que pode projetar uma sombra", "tipo_esperado": "concreto"},
    {"termo": "criatura", "en": "a living thing that has (or can develop) the ability to act or function independently",
     "pt": "ser vivo que tem ou pode desenvolver a capacidade de agir ou funcionar independentemente", "tipo_esperado": "concreto"},
    {"termo": "bentos", "en": "organisms (plants and animals) that live at or near the bottom of a sea",
     "pt": "organismos (plantas e animais) que vivem no fundo do mar ou perto dele", "tipo_esperado": "concreto"},
    {"termo": "célula", "en": "(biology) the basic structural and functional unit of all organisms",
     "pt": "(biologia) a unidade estrutural e funcional básica de todos os organismos", "tipo_esperado": "concreto"},
    {"termo": "animal", "en": "a living organism characterized by voluntary movement",
     "pt": "organismo vivo caracterizado pelo movimento voluntário", "tipo_esperado": "concreto"},
    {"termo": "artefato", "en": "a man-made object taken as a whole",
     "pt": "um objeto fabricado pelo homem considerado como um todo", "tipo_esperado": "concreto"},
    # abstratos
    {"termo": "abstração", "en": "a general concept formed by extracting common features from specific examples",
     "pt": "conceito geral formado pela extração de características comuns de exemplos específicos", "tipo_esperado": "abstrato"},
    {"termo": "condição", "en": "the way something is with respect to its main attributes",
     "pt": "o modo como algo se encontra em relação aos seus principais atributos", "tipo_esperado": "abstrato"},
    {"termo": "emoção", "en": "the experiencing of affective and emotional states",
     "pt": "a vivência de estados afetivos e emocionais", "tipo_esperado": "abstrato"},
    {"termo": "cognição", "en": "the psychological result of perception and learning and reasoning",
     "pt": "o resultado psicológico da percepção, do aprendizado e do raciocínio", "tipo_esperado": "abstrato"},
    {"termo": "magnitude", "en": "how much there is or how many there are of something that you can quantify",
     "pt": "a quantidade ou o número de algo que pode ser quantificado", "tipo_esperado": "abstrato"},
    {"termo": "relação social", "en": "a relation between living organisms (especially between people)",
     "pt": "uma relação entre organismos vivos (especialmente entre pessoas)", "tipo_esperado": "abstrato"},
]


class PromptAdaptativo(BaseModel):
    classificacao: str = Field(description="'concreto' se o conceito tem forma física direta, 'abstrato' se não tem")
    raciocinio: str = Field(description="1 frase explicando a escolha da cena/metáfora")
    prompt_visual: str = Field(description="Prompt em inglês pronto pro gerador de imagem. Se concreto: descrição literal do objeto/cena. Se abstrato: descrição de uma cena ou objeto CONCRETO que representa o significado por metáfora visual — NUNCA use a palavra abstrata como sujeito do prompt.")


def gerar_prompt_adaptativo(client, termo, definicao_en):
    instrucao = f"""Word: {termo}
Definition: {definicao_en}

You are creating a prompt for an AI image generator to illustrate this dictionary concept.

Step 1: Decide if this concept is CONCRETE (has a direct physical form you can draw literally,
like "animal" or "object") or ABSTRACT (a category, quality, relation, or mental/social concept,
like "condition" or "emotion", with no single physical form).

Step 2:
- If CONCRETE: write a literal, direct visual description of the concept.
- If ABSTRACT: do NOT try to draw the abstract word itself. Instead, think of ONE concrete
  scene, object, or pair of contrasting concrete objects that VISUALLY SYMBOLIZES the meaning
  of the definition (a visual metaphor). Describe that concrete scene instead.

The final prompt must describe a concrete, drawable scene either way. No text or letters in
the image. Square composition, clean illustration style."""
    resp = client.models.generate_content(
        model=MODELO_TEXTO, contents=instrucao,
        config={"response_mime_type": "application/json", "response_schema": PromptAdaptativo},
    )
    return PromptAdaptativo.model_validate_json(resp.text)


def gerar_imagem_pollinations(prompt_texto, caminho_saida, chave_pollinations, seed=42, modelo="flux"):
    prompt_codificado = urllib.parse.quote(prompt_texto)
    url = f"https://gen.pollinations.ai/image/{prompt_codificado}?model={modelo}&width=768&height=768&seed={seed}&nologo=true"
    headers = {"Authorization": f"Bearer {chave_pollinations}"}
    resp = requests.get(url, headers=headers, timeout=90)
    resp.raise_for_status()
    with open(caminho_saida, "wb") as f:
        f.write(resp.content)
    return caminho_saida


def gerar_imagem_huggingface(prompt_texto, caminho_saida, chave_hf, modelo="black-forest-labs/FLUX.1-schnell"):
    from huggingface_hub import InferenceClient
    client_hf = InferenceClient(provider="auto", api_key=chave_hf)
    imagem_pil = client_hf.text_to_image(prompt_texto, model=modelo)  # retorna um objeto PIL.Image
    imagem_pil.save(caminho_saida)
    return caminho_saida


def gerar_imagem_replicate(prompt_texto, caminho_saida, chave_replicate, modelo="black-forest-labs/flux-schnell"):
    import replicate
    client_replicate = replicate.Client(api_token=chave_replicate)
    output = client_replicate.run(modelo, input={"prompt": prompt_texto})
    # output é uma lista de FileOutput; pegamos o primeiro
    arquivo = output[0] if isinstance(output, list) else output
    with open(caminho_saida, "wb") as f:
        f.write(arquivo.read())
    return caminho_saida


# --- Execução ---
drive.mount('/content/drive')
PASTA_TESTE = "/content/drive/MyDrive/Grupo 3 - RAGI 2026/Teste_Prompts_Imagem"
os.makedirs(PASTA_TESTE, exist_ok=True)

gemini_key = getpass.getpass("Cole sua API key do Google AI Studio: ")
client = genai.Client(api_key=gemini_key)

print("\nAs chaves abaixo são opcionais — deixe em branco (Enter) pra pular um provedor.")
pollinations_key = getpass.getpass("Chave do Pollinations (enter.pollinations.ai) [Enter p/ pular]: ")
hf_key = getpass.getpass("Chave do Hugging Face (huggingface.co/settings/tokens) [Enter p/ pular]: ")
replicate_key = getpass.getpass("Chave do Replicate (replicate.com/account/api-tokens) [Enter p/ pular]: ")

PROVEDORES = []
if pollinations_key:
    PROVEDORES.append(("pollinations", lambda p, c: gerar_imagem_pollinations(p, c, pollinations_key)))
if hf_key:
    PROVEDORES.append(("huggingface", lambda p, c: gerar_imagem_huggingface(p, c, hf_key)))
if replicate_key:
    PROVEDORES.append(("replicate", lambda p, c: gerar_imagem_replicate(p, c, replicate_key)))

if not PROVEDORES:
    raise SystemExit("Nenhuma chave informada — precisa de pelo menos 1 provedor pra rodar o teste.")

print(f"\nProvedores ativos neste teste: {[nome for nome, _ in PROVEDORES]}")

resultados = []

for item in AMOSTRA_TESTE:
    termo_slug = item["termo"].replace(" ", "_")
    print(f"\n{'='*70}\n{item['termo']} (esperado: {item['tipo_esperado']})\n{'='*70}")

    # 1 prompt só, pela estratégia adaptativa (já validada como a melhor abordagem)
    r_adapt = gerar_prompt_adaptativo(client, item["termo"], item["en"])
    print(f"[prompt] classificado como: {r_adapt.classificacao} | {r_adapt.prompt_visual}")

    imagens_geradas = {}
    for nome_provedor, funcao_gerar in PROVEDORES:
        caminho = os.path.join(PASTA_TESTE, f"{termo_slug}__{nome_provedor}.png")
        try:
            funcao_gerar(r_adapt.prompt_visual, caminho)
            imagens_geradas[nome_provedor] = caminho
            print(f"   ✓ {nome_provedor}: ok")
        except Exception as e:
            print(f"   ✗ {nome_provedor}: falhou — {str(e)[:200]}")
            imagens_geradas[nome_provedor] = None
        time.sleep(3)

    resultados.append({
        "termo": item["termo"], "tipo_esperado": item["tipo_esperado"], "pt": item["pt"],
        "classificacao": r_adapt.classificacao, "prompt": r_adapt.prompt_visual,
        "imagens": imagens_geradas,
    })

with open(os.path.join(PASTA_TESTE, "resultados_teste.json"), "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# --- Galeria HTML: 1 linha por conceito, 1 coluna por provedor ---
nomes_provedores = [nome for nome, _ in PROVEDORES]

linhas_html = []
for r in resultados:
    celulas_imagem = []
    for nome_provedor in nomes_provedores:
        caminho = r["imagens"].get(nome_provedor)
        if caminho:
            celulas_imagem.append(f'<td><img src="{os.path.basename(caminho)}" width="280"><br><small>{nome_provedor}</small></td>')
        else:
            celulas_imagem.append(f'<td>(falhou: {nome_provedor})</td>')

    linhas_html.append(f"""
    <tr>
      <td><b>{r['termo']}</b><br><small>{r['pt']}</small><br>
          <i>[{r['tipo_esperado']} → classificado: {r['classificacao']}]</i><br>
          <small>{r['prompt']}</small></td>
      {''.join(celulas_imagem)}
    </tr>""")

cabecalho_colunas = "".join(f"<th>{nome}</th>" for nome in nomes_provedores)

html = f"""<html><head><meta charset="utf-8"><style>
table {{ border-collapse: collapse; width: 100%; }}
td, th {{ border: 1px solid #ccc; padding: 10px; vertical-align: top; }}
</style></head><body>
<h2>Comparação entre provedores de geração de imagem</h2>
<table>
<tr><th>Conceito / Prompt</th>{cabecalho_colunas}</tr>
{''.join(linhas_html)}
</table></body></html>"""

with open(os.path.join(PASTA_TESTE, "galeria_comparacao.html"), "w", encoding="utf-8") as f:
    f.write(html)

print(f"\n\nConcluído. Abra {PASTA_TESTE}/galeria_comparacao.html no navegador (via Drive) pra comparar lado a lado.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cole sua API key do Google AI Studio: ··········

As chaves abaixo são opcionais — deixe em branco (Enter) pra pular um provedor.
Chave do Pollinations (enter.pollinations.ai) [Enter p/ pular]: ··········
Chave do Hugging Face (huggingface.co/settings/tokens) [Enter p/ pular]: ··········
Chave do Replicate (replicate.com/account/api-tokens) [Enter p/ pular]: ··········

Provedores ativos neste teste: ['huggingface']

objeto (esperado: concreto)
[prompt] classificado como: concreto | A single wooden spinning top resting on a clean wooden table, casting a distinct sharp shadow under bright warm studio lighting, minimalist composition, square format, clean illustration style
   ✓ huggingface: ok

criatura (esperado: concreto)
[prompt] classificado como: concreto | A whimsical fantasy creature standing in a lush enchanted forest, detailed fur and glowing eyes, sq

# Teste Final

In [ ]:
!pip install -q -U openai huggingface_hub pillow pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 28.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.


In [ ]:
import os
import re
import json
import time
import getpass
from datetime import datetime

from google.colab import drive

from openai import OpenAI
from huggingface_hub import InferenceClient
from PIL import Image

print("=" * 75)
print("RAGI 2026 — GERAÇÃO DEFINITIVA DE IMAGENS DOS CONCEITOS")
print("=" * 75)

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

BASE_RAGI = "/content/drive/MyDrive/Grupo 3 - RAGI 2026"

ARQUIVO_CONCEITOS = os.path.join(
    BASE_RAGI,
    "Traducao",
    "conceitos_traduzidos.json"
)

PASTA_IMAGENS = os.path.join(
    BASE_RAGI,
    "Imagens_Conceitos"
)

PASTA_METADADOS = os.path.join(
    BASE_RAGI,
    "Imagens_Conceitos",
    "metadados"
)

PASTA_GALERIA = os.path.join(
    BASE_RAGI,
    "Imagens_Conceitos",
    "galeria"
)

ARQUIVO_PROGRESSO = os.path.join(
    PASTA_METADADOS,
    "progresso_imagens_conceitos.json"
)

ARQUIVO_RESULTADOS = os.path.join(
    PASTA_METADADOS,
    "resultados_imagens_conceitos.json"
)

os.makedirs(PASTA_IMAGENS, exist_ok=True)
os.makedirs(PASTA_METADADOS, exist_ok=True)
os.makedirs(PASTA_GALERIA, exist_ok=True)

print("Drive montado.")
print()
print("Pasta principal:")
print(BASE_RAGI)
print()
print("Pasta das imagens:")
print(PASTA_IMAGENS)
print()
print("Checkpoint:")
print(ARQUIVO_PROGRESSO)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive montado.

Pasta principal:
/content/drive/MyDrive/Grupo 3 - RAGI 2026

Pasta das imagens:
/content/drive/MyDrive/Grupo 3 - RAGI 2026/Imagens_Conceitos

Checkpoint:
/content/drive/MyDrive/Grupo 3 - RAGI 2026/Imagens_Conceitos/metadados/progresso_imagens_conceitos.json


In [ ]:
# ============================================================
# CONFIGURAÇÃO DA PRODUÇÃO
# ============================================================

LIMITE_CONCEITOS = 50

# True = processa os primeiros LIMITE_CONCEITOS do arquivo.
# False = tenta processar todos.
PROCESSAR_APENAS_LIMITE = True

# Modelo de planejamento visual do DeepSeek.
MODELO_DEEPSEEK = "deepseek-chat"

# Modelo de geração de imagens no Hugging Face.
MODELO_HF = "black-forest-labs/FLUX.1-schnell"

# Dimensão desejada.
LARGURA = 768
ALTURA = 768

# Número máximo de tentativas por chamada.
MAX_TENTATIVAS = 4

# Intervalo entre chamadas.
INTERVALO_DEEPEEK = 1.5
INTERVALO_HF = 3.0

print("Configuração:")
print("  DeepSeek:", MODELO_DEEPSEEK)
print("  Hugging Face:", MODELO_HF)
print("  Tamanho:", f"{LARGURA}x{ALTURA}")
print("  Limite:", LIMITE_CONCEITOS)

Configuração:
  DeepSeek: deepseek-chat
  Hugging Face: black-forest-labs/FLUX.1-schnell
  Tamanho: 768x768
  Limite: 50


In [ ]:
print("=" * 75)
print("AUTENTICAÇÃO")
print("=" * 75)

DEEPSEEK_KEY = getpass.getpass(
    "Cole sua API key do DeepSeek: "
)

HF_KEY = getpass.getpass(
    "Cole seu token do Hugging Face: "
)

if not DEEPSEEK_KEY:
    raise RuntimeError("A chave do DeepSeek não foi informada.")

if not HF_KEY:
    raise RuntimeError("O token do Hugging Face não foi informado.")

print()
print("✅ Chaves recebidas.")

AUTENTICAÇÃO


In [ ]:
from openai import OpenAI
from huggingface_hub import InferenceClient

deepseek_client = OpenAI(
    api_key=DEEPSEEK_KEY,
    base_url="https://api.deepseek.com"
)

hf_client = InferenceClient(
    provider="auto",
    api_key=HF_KEY
)

print("✅ DeepSeek conectado.")
print("✅ Hugging Face conectado.")

✅ DeepSeek conectado.
✅ Hugging Face conectado.


In [ ]:
print("=" * 75)
print("CARREGANDO CONCEITOS")
print("=" * 75)

if not os.path.exists(ARQUIVO_CONCEITOS):
    raise FileNotFoundError(
        f"Arquivo não encontrado:\n{ARQUIVO_CONCEITOS}"
    )

with open(
    ARQUIVO_CONCEITOS,
    "r",
    encoding="utf-8"
) as f:
    conceitos = json.load(f)

print(f"Registros encontrados: {len(conceitos):,}")

if PROCESSAR_APENAS_LIMITE:
    itens_trabalho = conceitos[:LIMITE_CONCEITOS]
else:
    itens_trabalho = conceitos

print(f"Registros selecionados para produção: {len(itens_trabalho):,}")

print()
print("Primeiro conceito:")
print(json.dumps(itens_trabalho[0], ensure_ascii=False, indent=2))

CARREGANDO CONCEITOS
Registros encontrados: 39,937
Registros selecionados para produção: 50

Primeiro conceito:
{
  "id": "omw-pt-00001740-n",
  "id_interno_ooinfo": "cmt1k4jopusg035btslfb3dc0",
  "palavra": "ente",
  "classe_gramatical": "Substantivo",
  "definicao_en": "that which is perceived or known or inferred to have its own distinct existence (living or nonliving)",
  "definicao_pt": "aquilo que é percebido, conhecido ou inferido como detentor de uma existência própria distinta, seja vivente ou não vivente",
  "confianca": 0.98,
  "observacao": "",
  "modelo": "gemini-3.5-flash-lite",
  "timestamp": "2026-08-24T19:23:56.180706"
}


In [ ]:
print("=" * 75)
print("CARREGANDO CONCEITOS")
print("=" * 75)

if not os.path.exists(ARQUIVO_CONCEITOS):
    raise FileNotFoundError(
        f"Arquivo não encontrado:\n{ARQUIVO_CONCEITOS}"
    )

with open(
    ARQUIVO_CONCEITOS,
    "r",
    encoding="utf-8"
) as f:
    conceitos = json.load(f)

print(f"Registros encontrados: {len(conceitos):,}")

if PROCESSAR_APENAS_LIMITE:
    itens_trabalho = conceitos[:LIMITE_CONCEITOS]
else:
    itens_trabalho = conceitos

print(f"Registros selecionados para produção: {len(itens_trabalho):,}")

print()
print("Primeiro conceito:")
print(json.dumps(itens_trabalho[0], ensure_ascii=False, indent=2))

CARREGANDO CONCEITOS
Registros encontrados: 39,937
Registros selecionados para produção: 50

Primeiro conceito:
{
  "id": "omw-pt-00001740-n",
  "id_interno_ooinfo": "cmt1k4jopusg035btslfb3dc0",
  "palavra": "ente",
  "classe_gramatical": "Substantivo",
  "definicao_en": "that which is perceived or known or inferred to have its own distinct existence (living or nonliving)",
  "definicao_pt": "aquilo que é percebido, conhecido ou inferido como detentor de uma existência própria distinta, seja vivente ou não vivente",
  "confianca": 0.98,
  "observacao": "",
  "modelo": "gemini-3.5-flash-lite",
  "timestamp": "2026-08-24T19:23:56.180706"
}


In [ ]:
def slugify(texto):
    texto = str(texto).strip().lower()

    texto = re.sub(
        r"[^\w\s-]",
        "",
        texto,
        flags=re.UNICODE
    )

    texto = re.sub(
        r"[\s]+",
        "_",
        texto
    )

    texto = re.sub(
        r"_+",
        "_",
        texto
    )

    return texto[:150]


def nome_arquivo_conceito(item):
    termo = item.get("palavra") or item.get("termo") or "conceito"

    return slugify(termo)

In [ ]:
SYSTEM_PROMPT = """
You are an expert visual semantic designer.

Your job is to transform dictionary definitions into highly
recognizable visual representations for an educational
concept dictionary.

The generated image will represent ONE dictionary concept.

Your priority is SEMANTIC CLARITY, not artistic complexity.

RULES:

1. Read the English and Portuguese definitions carefully.

2. Determine whether the concept is:
   - "concreto": it has a direct physical or observable referent;
   - "abstrato": it represents a quality, state, relation,
     mental process, quantity, category, condition, or other
     concept without one unique physical form.

3. For CONCRETE concepts:
   - represent the actual referent;
   - choose the most prototypical visual example;
   - do not choose an unnecessarily exotic example;
   - make the concept immediately recognizable.

4. For ABSTRACT concepts:
   - do NOT attempt to draw the word itself;
   - do NOT create generic glowing shapes;
   - create ONE concrete visual metaphor or observable situation
     that communicates the dictionary definition;
   - the metaphor must be easy to understand;
   - prefer common visual conventions and everyday scenes.

5. The image must communicate the DEFINITION, not merely the
   word.

6. Avoid ambiguous symbolism.

7. Avoid unnecessary surrealism.

8. Avoid multiple unrelated concepts in one image.

9. Avoid written language.

10. Never put the concept name, labels, captions, letters,
    equations or words inside the image.

11. The final image should look like a clean educational
    encyclopedia illustration.

12. Use a simple square composition.

13. Use a clear main subject.

14. Use realistic or coherent spatial relationships.

15. The final image prompt MUST be written in English.

16. The final prompt should directly describe what the image
    generator must draw. Do not explain the reasoning inside
    the final prompt.

17. For scientific concepts, use scientifically recognizable
    visual structures when appropriate.

18. For abstract concepts, use concrete visual metaphors,
    human actions, objects, relationships, comparisons or
    observable situations.

19. Never use the Portuguese word as the subject of the image.

20. Do not include artistic instructions that overpower the
    semantic meaning.

Return ONLY valid JSON.
"""

In [ ]:
def gerar_planejamento_visual(item):

    termo = item.get("palavra") or item.get("termo") or ""

    definicao_en = (
        item.get("definicao_en")
        or item.get("descricao_en")
        or ""
    )

    definicao_pt = (
        item.get("definicao_pt")
        or item.get("descricao_pt")
        or ""
    )

    user_prompt = f"""
Concept term:
{termo}

English dictionary definition:
{definicao_en}

Portuguese definition:
{definicao_pt}

Create the visual representation for this dictionary concept.

Return exactly this JSON structure:

{{
  "classificacao": "concreto",
  "ideia_visual": "short explanation of the visual concept",
  "sujeito_principal": "main concrete subject",
  "acao_ou_relacao": "main action or relationship",
  "prompt_visual": "final English image-generation prompt"
}}

The field "classificacao" must be exactly:
"concreto" or "abstrato".

The field "prompt_visual" must contain ONLY the final
description for the image generator.
"""

    ultimo_erro = None

    for tentativa in range(1, MAX_TENTATIVAS + 1):

        try:

            response = deepseek_client.chat.completions.create(
                model=MODELO_DEEPSEEK,
                messages=[
                    {
                        "role": "system",
                        "content": SYSTEM_PROMPT
                    },
                    {
                        "role": "user",
                        "content": user_prompt
                    }
                ],
                response_format={
                    "type": "json_object"
                },
                temperature=0.4,
                max_tokens=900
            )

            texto = response.choices[0].message.content

            resultado = json.loads(texto)

            classificacao = resultado.get("classificacao", "").lower().strip()

            if classificacao not in ["concreto", "abstrato"]:
                raise ValueError(
                    f"classificacao inválida: {classificacao}"
                )

            prompt_visual = resultado.get(
                "prompt_visual",
                ""
            ).strip()

            if not prompt_visual:
                raise ValueError(
                    "DeepSeek retornou prompt_visual vazio."
                )

            return resultado

        except Exception as e:

            ultimo_erro = e

            print(
                f"   ⚠️ DeepSeek tentativa "
                f"{tentativa}/{MAX_TENTATIVAS}: {e}"
            )

            if tentativa < MAX_TENTATIVAS:
                time.sleep(3 * tentativa)

    raise RuntimeError(
        f"DeepSeek falhou após {MAX_TENTATIVAS} "
        f"tentativas: {ultimo_erro}"
    )

In [ ]:
def gerar_imagem_huggingface(
    prompt,
    caminho_saida
):

    ultimo_erro = None

    for tentativa in range(1, MAX_TENTATIVAS + 1):

        try:

            imagem = hf_client.text_to_image(
                prompt=prompt,
                model=MODELO_HF
            )

            if imagem is None:
                raise RuntimeError(
                    "Hugging Face retornou imagem vazia."
                )

            imagem.save(caminho_saida)

            if not os.path.exists(caminho_saida):
                raise RuntimeError(
                    "Arquivo de imagem não foi criado."
                )

            tamanho = os.path.getsize(caminho_saida)

            if tamanho < 1000:
                raise RuntimeError(
                    f"Imagem suspeita: apenas {tamanho} bytes."
                )

            return caminho_saida

        except Exception as e:

            ultimo_erro = e

            print(
                f"   ⚠️ Hugging Face tentativa "
                f"{tentativa}/{MAX_TENTATIVAS}: {e}"
            )

            if tentativa < MAX_TENTATIVAS:
                time.sleep(5 * tentativa)

    raise RuntimeError(
        f"Hugging Face falhou após "
        f"{MAX_TENTATIVAS} tentativas: {ultimo_erro}"
    )

In [ ]:
def extrair_dados_conceito(item):

    return {
        "id": item.get("id"),
        "id_interno_ooinfo": item.get(
            "id_interno_ooinfo"
        ),
        "palavra": item.get("palavra") or item.get("termo"),
        "classe_gramatical": item.get(
            "classe_gramatical"
        ),
        "definicao_en": item.get(
            "definicao_en"
        ),
        "definicao_pt": item.get(
            "definicao_pt"
        )
    }

In [ ]:
import re

print("=" * 75)
print("INICIANDO PRODUÇÃO")
print("=" * 75)

progresso = carregar_json(ARQUIVO_PROGRESSO, {})  # nosso_id -> {"status": "ok"|"falha", ...}

total = len(itens_trabalho)

sucessos = sum(
    1
    for r in progresso.values()
    if r.get("status") == "ok"
)

print(
    f"Já concluídos anteriormente: "
    f"{sucessos}/{total}"
)

print()

inicio_total = time.time()

for indice, item in enumerate(
    itens_trabalho,
    start=1
):

    dados = extrair_dados_conceito(item)

    conceito_id = dados["id"]
    termo = dados["palavra"]

    if not conceito_id:
        print(
            f"[{indice}/{total}] ⚠️ "
            f"Conceito sem ID. Pulando."
        )
        continue

    # ========================================================
    # CHECKPOINT
    # ========================================================

    registro_existente = progresso.get(
        conceito_id,
        {}
    )

    if registro_existente.get("status") == "ok":

        print(
            f"[{indice}/{total}] "
            f"{termo} — já concluído. Pulando."
        )

        continue

    print()
    print("=" * 75)
    print(
        f"[{indice}/{total}] {termo}"
    )
    print("=" * 75)

    try:

        # ====================================================
        # 1. DEEPSEEK
        # ====================================================

        print("🧠 Criando planejamento visual...")

        planejamento = gerar_planejamento_visual(
            item
        )

        classificacao = planejamento[
            "classificacao"
        ]

        ideia_visual = planejamento.get(
            "ideia_visual",
            ""
        )

        sujeito = planejamento.get(
            "sujeito_principal",
            ""
        )

        acao = planejamento.get(
            "acao_ou_relacao",
            ""
        )

        prompt_visual = planejamento[
            "prompt_visual"
        ]

        print(
            f"   Classificação: {classificacao}"
        )

        print(
            f"   Ideia: {ideia_visual}"
        )

        print(
            f"   Sujeito: {sujeito}"
        )

        print(
            f"   Ação/relação: {acao}"
        )

        print(
            f"   Prompt: {prompt_visual}"
        )

        time.sleep(INTERVALO_DEEPEEK)

        # ====================================================
        # 2. HUGGING FACE
        # ====================================================

        print("🎨 Gerando imagem no Hugging Face...")

        nome = nome_arquivo_conceito(item)

        caminho_imagem = os.path.join(
            PASTA_IMAGENS,
            f"{nome}.png"
        )

        # Evita colisão de nomes
        if os.path.exists(caminho_imagem):

            caminho_imagem = os.path.join(
                PASTA_IMAGENS,
                f"{nome}__{conceito_id}.png"
            )

        gerar_imagem_huggingface(
            prompt_visual,
            caminho_imagem
        )

        print(
            f"   ✅ Imagem salva:"
        )

        print(
            f"   {caminho_imagem}"
        )

        time.sleep(INTERVALO_HF)

        # ====================================================
        # 3. REGISTRAR SUCESSO
        # ====================================================

        progresso[conceito_id] = {

            "status": "ok",

            "id": conceito_id,

            "id_interno_ooinfo":
                dados["id_interno_ooinfo"],

            "palavra": termo,

            "classe_gramatical":
                dados["classe_gramatical"],

            "definicao_en":
                dados["definicao_en"],

            "definicao_pt":
                dados["definicao_pt"],

            "classificacao":
                classificacao,

            "ideia_visual":
                ideia_visual,

            "sujeito_principal":
                sujeito,

            "acao_ou_relacao":
                acao,

            "prompt_visual":
                prompt_visual,

            "modelo_texto":
                MODELO_DEEPSEEK,

            "modelo_imagem":
                MODELO_HF,

            "arquivo_imagem":
                caminho_imagem,

            "timestamp":
                datetime.now().isoformat()
        }

    except Exception as e:

        # ====================================================
        # REGISTRAR FALHA
        # ====================================================

        progresso[conceito_id] = {

            "status": "falha",

            "id": conceito_id,

            "id_interno_ooinfo":
                dados["id_interno_ooinfo"],

            "palavra": termo,

            "erro": str(e),

            "timestamp":
                datetime.now().isoformat()
        }

        print(
            f"   ❌ FALHA: {e}"
        )

    # ========================================================
    # CHECKPOINT IMEDIATO
    # ========================================================

    salvar_json_seguro(
        ARQUIVO_PROGRESSO,
        progresso
    )

    print(
        "   💾 Checkpoint salvo."
    )


tempo_total = (
    time.time() - inicio_total
)

print()
print("=" * 75)
print("PRODUÇÃO FINALIZADA")
print("=" * 75)

sucessos = sum(
    1
    for r in progresso.values()
    if r.get("status") == "ok"
)

falhas = sum(
    1
    for r in progresso.values()
    if r.get("status") == "falha"
)

print(
    f"Sucesso: {sucessos}/{total}"
)

print(
    f"Falhas:  {falhas}/{total}"
)

print(
    f"Tempo desta execução: "
    f"{tempo_total / 60:.1f} minutos"
)

print()
print(
    f"Checkpoint:\n{ARQUIVO_PROGRESSO}"
)

INICIANDO PRODUÇÃO
Já concluídos anteriormente: 0/50


[1/50] ente
🧠 Criando planejamento visual...
   Classificação: abstrato
   Ideia: A single tree standing alone in a vast, empty landscape, representing the concept of a distinct entity with its own existence.
   Sujeito: a lone tree
   Ação/relação: standing alone in a vast landscape
   Prompt: A single, mature tree standing alone in the center of a vast, flat, empty landscape under a clear sky. The tree is clearly defined, with a strong trunk and full canopy. The scene is simple, realistic, and emphasizes the tree as the sole distinct object in the environment. No text, no other objects, no people. Educational encyclopedia illustration style, clean and uncluttered, square composition.
🎨 Gerando imagem no Hugging Face...
   ✅ Imagem salva:
   /content/drive/MyDrive/Grupo 3 - RAGI 2026/Imagens_Conceitos/ente.png
   💾 Checkpoint salvo.

[2/50] entidade física
🧠 Criando planejamento visual...
   Classificação: concreto
   Ideia: A roc

In [ ]:
print("=" * 75)
print("INICIANDO PRODUÇÃO")
print("=" * 75)

total = len(itens_trabalho)

sucessos = sum(
    1
    for r in progresso.values()
    if r.get("status") == "ok"
)

print(
    f"Já concluídos anteriormente: "
    f"{sucessos}/{total}"
)

print()

inicio_total = time.time()

for indice, item in enumerate(
    itens_trabalho,
    start=1
):

    dados = extrair_dados_conceito(item)

    conceito_id = dados["id"]
    termo = dados["palavra"]

    if not conceito_id:
        print(
            f"[{indice}/{total}] ⚠️ "
            f"Conceito sem ID. Pulando."
        )
        continue

    # ========================================================
    # CHECKPOINT
    # ========================================================

    registro_existente = progresso.get(
        conceito_id,
        {}
    )

    if registro_existente.get("status") == "ok":

        print(
            f"[{indice}/{total}] "
            f"{termo} — já concluído. Pulando."
        )

        continue

    print()
    print("=" * 75)
    print(
        f"[{indice}/{total}] {termo}"
    )
    print("=" * 75)

    try:

        # ====================================================
        # 1. DEEPSEEK
        # ====================================================

        print("🧠 Criando planejamento visual...")

        planejamento = gerar_planejamento_visual(
            item
        )

        classificacao = planejamento[
            "classificacao"
        ]

        ideia_visual = planejamento.get(
            "ideia_visual",
            ""
        )

        sujeito = planejamento.get(
            "sujeito_principal",
            ""
        )

        acao = planejamento.get(
            "acao_ou_relacao",
            ""
        )

        prompt_visual = planejamento[
            "prompt_visual"
        ]

        print(
            f"   Classificação: {classificacao}"
        )

        print(
            f"   Ideia: {ideia_visual}"
        )

        print(
            f"   Sujeito: {sujeito}"
        )

        print(
            f"   Ação/relação: {acao}"
        )

        print(
            f"   Prompt: {prompt_visual}"
        )

        time.sleep(INTERVALO_DEEPEEK)

        # ====================================================
        # 2. HUGGING FACE
        # ====================================================

        print("🎨 Gerando imagem no Hugging Face...")

        nome = nome_arquivo_conceito(item)

        caminho_imagem = os.path.join(
            PASTA_IMAGENS,
            f"{nome}.png"
        )

        # Evita colisão de nomes
        if os.path.exists(caminho_imagem):

            caminho_imagem = os.path.join(
                PASTA_IMAGENS,
                f"{nome}__{conceito_id}.png"
            )

        gerar_imagem_huggingface(
            prompt_visual,
            caminho_imagem
        )

        print(
            f"   ✅ Imagem salva:"
        )

        print(
            f"   {caminho_imagem}"
        )

        time.sleep(INTERVALO_HF)

        # ====================================================
        # 3. REGISTRAR SUCESSO
        # ====================================================

        progresso[conceito_id] = {

            "status": "ok",

            "id": conceito_id,

            "id_interno_ooinfo":
                dados["id_interno_ooinfo"],

            "palavra": termo,

            "classe_gramatical":
                dados["classe_gramatical"],

            "definicao_en":
                dados["definicao_en"],

            "definicao_pt":
                dados["definicao_pt"],

            "classificacao":
                classificacao,

            "ideia_visual":
                ideia_visual,

            "sujeito_principal":
                sujeito,

            "acao_ou_relacao":
                acao,

            "prompt_visual":
                prompt_visual,

            "modelo_texto":
                MODELO_DEEPSEEK,

            "modelo_imagem":
                MODELO_HF,

            "arquivo_imagem":
                caminho_imagem,

            "timestamp":
                datetime.now().isoformat()
        }

    except Exception as e:

        # ====================================================
        # REGISTRAR FALHA
        # ====================================================

        progresso[conceito_id] = {

            "status": "falha",

            "id": conceito_id,

            "id_interno_ooinfo":
                dados["id_interno_ooinfo"],

            "palavra": termo,

            "erro": str(e),

            "timestamp":
                datetime.now().isoformat()
        }

        print(
            f"   ❌ FALHA: {e}"
        )

    # ========================================================
    # CHECKPOINT IMEDIATO
    # ========================================================

    salvar_json_seguro(
        ARQUIVO_PROGRESSO,
        progresso
    )

    print(
        "   💾 Checkpoint salvo."
    )


tempo_total = (
    time.time() - inicio_total
)

print()
print("=" * 75)
print("PRODUÇÃO FINALIZADA")
print("=" * 75)

sucessos = sum(
    1
    for r in progresso.values()
    if r.get("status") == "ok"
)

falhas = sum(
    1
    for r in progresso.values()
    if r.get("status") == "falha"
)

print(
    f"Sucesso: {sucessos}/{total}"
)

print(
    f"Falhas:  {falhas}/{total}"
)

print(
    f"Tempo desta execução: "
    f"{tempo_total / 60:.1f} minutos"
)

print()
print(
    f"Checkpoint:\n{ARQUIVO_PROGRESSO}"
)

INICIANDO PRODUÇÃO
Já concluídos anteriormente: 16/50

[1/50] ente — já concluído. Pulando.
[2/50] entidade física — já concluído. Pulando.
[3/50] abstração — já concluído. Pulando.
[4/50] objeto — já concluído. Pulando.
[5/50] todo — já concluído. Pulando.
[6/50] congênere — já concluído. Pulando.
[7/50] criatura — já concluído. Pulando.
[8/50] bentos — já concluído. Pulando.
[9/50] Heterotrofismo — já concluído. Pulando.

[10/50] vida
🧠 Criando planejamento visual...
   Classificação: concreto
   Ideia: A lush green landscape with a variety of living organisms including trees, flowers, birds, and a deer, symbolizing the collective of living beings.
   Sujeito: A diverse natural scene with multiple living organisms
   Ação/relação: Coexistence in a shared habitat
   Prompt: A vibrant, realistic illustration of a lush green meadow with a large oak tree, colorful wildflowers, a deer grazing, and birds flying in the sky. The scene is teeming with life, showing a variety of plants and ani

In [ ]:
resultados_finais = []

for item in itens_trabalho:

    conceito_id = item.get("id")

    if conceito_id in progresso:

        resultados_finais.append(
            progresso[conceito_id]
        )

salvar_json_seguro(
    ARQUIVO_RESULTADOS,
    resultados_finais
)

print(
    f"Resultados salvos em:\n"
    f"{ARQUIVO_RESULTADOS}"
)

Resultados salvos em:
/content/drive/MyDrive/Grupo 3 - RAGI 2026/Imagens_Conceitos/metadados/resultados_imagens_conceitos.json


In [ ]:
import os
from IPython.display import display, HTML

ARQUIVO_HTML = os.path.join(
    PASTA_GALERIA,
    "galeria_imagens_conceitos.html"
)

display(
    HTML(
        f"""
        <p>
        <a href="{ARQUIVO_HTML}"
           target="_blank">
           👉 Abrir galeria de imagens
        </a>
        </p>
        """
    )
)

In [ ]:
from html import escape

registros_ok = [
    r for r in progresso.values()
    if r.get("status") == "ok"
]

registros_ok.sort(
    key=lambda x: x.get("palavra", "")
)

linhas = []

for r in registros_ok:

    caminho = r.get(
        "arquivo_imagem",
        ""
    )

    if not caminho or not os.path.exists(caminho):
        continue

    nome_imagem = os.path.basename(
        caminho
    )

    termo = escape(
        str(r.get("palavra", ""))
    )

    classificacao = escape(
        str(r.get("classificacao", ""))
    )

    prompt = escape(
        str(r.get("prompt_visual", ""))
    )

    ideia = escape(
        str(r.get("ideia_visual", ""))
    )

    linhas.append(
        f"""
        <div class="card">

            <img src="../{nome_imagem}">

            <div class="info">

                <h2>{termo}</h2>

                <div class="tipo">
                    {classificacao}
                </div>

                <p>
                    <b>Ideia visual:</b><br>
                    {ideia}
                </p>

                <details>
                    <summary>Ver prompt</summary>
                    <p>{prompt}</p>
                </details>

            </div>

        </div>
        """
    )

html = f"""
<!DOCTYPE html>

<html lang="pt-BR">

<head>

<meta charset="UTF-8">

<title>
RAGI — Galeria de Imagens dos Conceitos
</title>

<style>

body {{
    font-family: Arial, sans-serif;
    background: #f4f4f4;
    margin: 0;
    padding: 30px;
}}

h1 {{
    text-align: center;
}}

.container {{
    display: grid;
    grid-template-columns:
        repeat(auto-fill, minmax(350px, 1fr));
    gap: 25px;
}}

.card {{
    background: white;
    border-radius: 10px;
    overflow: hidden;
    box-shadow:
        0 2px 8px rgba(0,0,0,0.12);
}}

.card img {{
    width: 100%;
    display: block;
}}

.info {{
    padding: 15px;
}}

.info h2 {{
    margin-top: 0;
}}

.tipo {{
    display: inline-block;
    padding: 5px 10px;
    background: #eee;
    border-radius: 5px;
    margin-bottom: 10px;
}}

details {{
    margin-top: 10px;
}}

summary {{
    cursor: pointer;
    font-weight: bold;
}}

</style>

</head>

<body>

<h1>
RAGI 2026 — Imagens dos Conceitos
</h1>

<p>
Total de imagens:
<b>{len(registros_ok)}</b>
</p>

<div class="container">

{''.join(linhas)}

</div>

</body>

</html>
"""

ARQUIVO_HTML = os.path.join(
    PASTA_GALERIA,
    "galeria_imagens_conceitos.html"
)

with open(
    ARQUIVO_HTML,
    "w",
    encoding="utf-8"
) as f:
    f.write(html)

print(
    "Galeria criada:"
)

print(
    ARQUIVO_HTML
)

Galeria criada:
/content/drive/MyDrive/Grupo 3 - RAGI 2026/Imagens_Conceitos/galeria/galeria_imagens_conceitos.html


# Codigo completo gerar e importar imagens

# Codigo Final V2 otimizado

In [ ]:
import os
import json
import time
import random
import base64
import hashlib
import threading
import urllib.parse
import requests
import shutil

from pathlib import Path
from datetime import datetime
from getpass import getpass
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import drive

# ================================================================
# 1. CONFIGURAÇÕES
# ================================================================

RAIZ_PROJETO = Path("/content/drive/MyDrive/Grupo 3 - RAGI 2026")
PASTA_WORDNET = RAIZ_PROJETO / "Datasets" / "Wordnet"
PASTA_IMAGENS = RAIZ_PROJETO / "Imagens" / "Conceitos"
PASTA_IMAGENS.mkdir(parents=True, exist_ok=True)

ARQUIVO_CONCEITOS = PASTA_WORDNET / "conceitos.json"
ARQUIVO_PALAVRAS = PASTA_WORDNET / "palavras.json"
ARQUIVO_MAPEAMENTO = PASTA_WORDNET / "mapeamento_conceitos_ooinfo.json"
ARQUIVO_TRADUCOES = RAIZ_PROJETO / "Traducao" / "conceitos_traduzidos.json"
ARQUIVO_PROGRESSO = PASTA_WORDNET / "progresso_imagens_conceitos.json"

BASE_URL = "https://ooinfo.org"
ID_LISTA_CONCEITOS = "cmsyyeh9d001b35btu2ginbbv"
CAMPO_IMAGEM = "imagem"

DEEPSEEK_URL = "https://api.deepseek.com/chat/completions"
DEEPSEEK_MODELO = "deepseek-v4-flash"

GEMINI_MODELOS = ["gemini-3.7-flash", "gemini-2.5-flash-lite"]

# Domínio LEGADO confirmado funcionando (o moderno devolveu 401 pra essa chave)
POLLINATIONS_URL = "https://image.pollinations.ai/prompt/"
POLLINATIONS_MODELO = "flux"
LARGURA_IMAGEM = 512
ALTURA_IMAGEM = 512

FILTRO_SUBSTANTIVO = False  # True = só Substantivo (~32.674). False = todos (~43.895).

NUM_WORKERS = 4
SALVAR_A_CADA = 50
MAX_TENTATIVAS_DEEPSEEK = 3
MAX_TENTATIVAS_GEMINI = 2
MAX_TENTATIVAS_POLLINATIONS = 10
MAX_TENTATIVAS_OOINFO = 3
TIMEOUT_DEEPSEEK = 45
TIMEOUT_GEMINI = 45
TIMEOUT_POLLINATIONS = (15, 120)
TIMEOUT_OOINFO = 60
BACKOFF_BASE = 2
BACKOFF_MAX = 30

# Limitador de taxa dedicado ao Pollinations: no máximo 1 requisição de cada
# vez, com esse intervalo mínimo entre chamadas -- independente de quantos
# workers estejam rodando ao mesmo tempo. Comece conservador; depois de rodar
# limpo por uns minutos sem 429, reduza aos poucos (ex: 2.0). Vale conferir
# também o dashboard em enter.pollinations.ai pra ver o limite real da chave.
POLLINATIONS_MIN_INTERVAL = 5.0
_pollinations_lock = threading.Lock()
_pollinations_proximo_horario = [0.0]


def _aguardar_vez_pollinations():
    with _pollinations_lock:
        agora_ts = time.time()
        espera = _pollinations_proximo_horario[0] - agora_ts
        if espera > 0:
            time.sleep(espera)
        _pollinations_proximo_horario[0] = max(agora_ts, _pollinations_proximo_horario[0]) + POLLINATIONS_MIN_INTERVAL


# ================================================================
# 2. DRIVE E ARQUIVOS
# ================================================================

# Desmonta e limpa o ponto de montagem do Drive para evitar `ValueError: Mountpoint must not already contain files`
if os.path.exists('/content/drive'):
    try:
        drive.flush_and_unmount()
        print("Google Drive desmontado com sucesso.")
    except ValueError:
        print("Google Drive não estava montado ou já foi desmontado.")
    except Exception as e:
        print(f"Erro ao tentar desmontar Google Drive: {e}")

# Garante que o diretório '/content/drive' esteja vazio.
if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    print("Limpando o diretório /content/drive...")
    for item in os.listdir('/content/drive'):
        item_path = os.path.join('/content/drive', item)
        try:
            if os.path.isfile(item_path) or os.path.islink(item_path):
                os.remove(item_path)
            elif os.path.isdir(item_path):
                shutil.rmtree(item_path)
            print(f"Removido: {item_path}")
        except Exception as e:
            print(f"Erro ao remover {item_path}: {e}")
    print("Diretório /content/drive limpo.")

drive.mount("/content/drive", force_remount=False)

for arquivo in (ARQUIVO_CONCEITOS, ARQUIVO_PALAVRAS, ARQUIVO_MAPEAMENTO):
    if not arquivo.exists():
        raise FileNotFoundError(f"Arquivo não encontrado:\n{arquivo}")

with open(ARQUIVO_CONCEITOS, "r", encoding="utf-8") as f:
    conceitos = json.load(f)
with open(ARQUIVO_PALAVRAS, "r", encoding="utf-8") as f:
    palavras = json.load(f)
with open(ARQUIVO_MAPEAMENTO, "r", encoding="utf-8") as f:
    mapeamento_conceitos = json.load(f)

traducoes_por_id = {}
if ARQUIVO_TRADUCOES.exists():
    with open(ARQUIVO_TRADUCOES, "r", encoding="utf-8") as f:
        traducoes = json.load(f)
    traducoes_por_id = {t["id"]: t.get("definicao_pt") for t in traducoes if t.get("definicao_pt")}

palavra_por_id = {p["id"]: p.get("palavra") or p.get("term") or p.get("label") or "" for p in palavras}

print("=" * 70)
print("BASE CARREGADA")
print("=" * 70)
print(f"Conceitos: {len(conceitos):,} | Traduções PT: {len(traducoes_por_id):,}")

# ================================================================
# 3. ESCOPO DE TRABALHO (com filtro por Substantivo)
# ================================================================

escopo = conceitos
if FILTRO_SUBSTANTIVO:
    escopo = [c for c in conceitos if c.get("classe_gramatical") == "Substantivo"]

itens_trabalho = []
sem_mapeamento = 0
sem_termo = 0

for c in escopo:
    nosso_id = c.get("id")
    if not nosso_id:
        continue
    id_interno = mapeamento_conceitos.get(nosso_id)
    if not id_interno:
        sem_mapeamento += 1
        continue

    termo = palavra_por_id.get(c.get("termo_preferencial"))
    if not termo:
        for pid in c.get("palavras") or []:
            if isinstance(pid, dict):
                pid = pid.get("id")
            candidato = palavra_por_id.get(pid)
            if candidato:
                termo = candidato
                break
    if not termo:
        termo = nosso_id
        sem_termo += 1

    itens_trabalho.append({
        "id": nosso_id,
        "id_interno": id_interno,
        "termo": termo,
        "descricao_en": (c.get("definicao") or "").strip(),
        "descricao_pt": (traducoes_por_id.get(nosso_id) or "").strip(),
        "classe_gramatical": c.get("classe_gramatical") or "",
    })

print(f"\nEscopo (FILTRO_SUBSTANTIVO={FILTRO_SUBSTANTIVO}): {len(escopo):,}")
print(f"Prontos para produção: {len(itens_trabalho):,}")
print(f"Sem mapeamento ooinfo: {sem_mapeamento:,} | Sem termo encontrado: {sem_termo:,}")

# ================================================================
# 4. CHECKPOINT
# ================================================================

def carregar_checkpoint():
    if not ARQUIVO_PROGRESSO.exists():
        return {}
    try:
        with open(ARQUIVO_PROGRESSO, "r", encoding="utf-8") as f:
            dados = json.load(f)
        return dados if isinstance(dados, dict) else {}
    except Exception as e:
        print(f"⚠️ Erro no checkpoint: {e}")
        return {}


def salvar_checkpoint():
    temporario = str(ARQUIVO_PROGRESSO) + ".tmp"
    with open(temporario, "w", encoding="utf-8") as f:
        json.dump(progresso, f, ensure_ascii=False, indent=2)
    os.replace(temporario, ARQUIVO_PROGRESSO)


def detectar_mime(conteudo):
    if conteudo[:3] == b"\xff\xd8\xff":
        return "image/jpeg"
    if conteudo[:8] == b"\x89PNG\r\n\x1a\n":
        return "image/png"
    if conteudo[:4] == b"RIFF":
        return "image/webp"
    return "image/png"


def imagem_valida(caminho):
    if not caminho:
        return False

    try:
        caminho = Path(caminho)
        return (
            caminho.exists()
            and caminho.is_file()
            and caminho.stat().st_size >= 10_000
        )
    except (TypeError, OSError):
        return False


progresso = carregar_checkpoint()

print("\n" + "=" * 70)
print("AUDITORIA DO CHECKPOINT")
print("=" * 70)

status_count = {}

for registro in progresso.values():
    status = registro.get("status", "sem_status")
    status_count[status] = status_count.get(status, 0) + 1

print(f"Registros no checkpoint: {len(progresso):,}")

for status, quantidade in sorted(status_count.items()):
    print(f"  {status}: {quantidade:,}")

imagens_checkpoint = 0
imagens_validas = 0

for registro in progresso.values():

    caminho = registro.get("imagem_path")

    if caminho:
        imagens_checkpoint += 1

        if imagem_valida(caminho):
            imagens_validas += 1

print(f"Registros com caminho de imagem: {imagens_checkpoint:,}")
print(f"Imagens válidas encontradas: {imagens_validas:,}")

print("=" * 70)

# ================================================================
# 5. LOGIN OOINFO E CHAVES
# ================================================================

EMAIL_OOINFO = input("\n📧 Email do ooinfo: ")
SENHA_OOINFO = getpass("🔐 Senha do ooinfo: ")


def fazer_login():
    resp = requests.post(
        f"{BASE_URL}/api/auth/login",
        json={"email": EMAIL_OOINFO, "password": SENHA_OOINFO}, timeout=30,
    )
    if resp.status_code not in (200, 201):
        raise RuntimeError("Falha no login ooinfo: " + resp.text[:500])
    token = resp.json().get("accessToken")
    if not token:
        raise RuntimeError("ooinfo não retornou accessToken.")
    return token


print("\n" + "=" * 70)
print("AUTENTICAÇÃO")
print("=" * 70)
TOKEN_OOINFO = fazer_login()
print("✓ Login ooinfo OK.")

DEEPSEEK_KEY = getpass("🔑 DeepSeek API Key: ")
GEMINI_KEY = getpass("🔑 Gemini API Key (fallback, Enter p/ pular): ")
POLLINATIONS_KEY = getpass("🔑 Pollinations API Key: ")
REPLICATE_KEY = getpass("🔑 Replicate API Key (fallback pago, ~US$0.003/img — Enter p/ pular): ")

token_lock = threading.Lock()


def obter_token():
    return TOKEN_OOINFO


def renovar_token():
    global TOKEN_OOINFO
    with token_lock:
        TOKEN_OOINFO = fazer_login()
        return TOKEN_OOINFO


# ================================================================
# 6. ESTADO GLOBAL, SESSÃO POR THREAD, UTILITÁRIOS
# ================================================================

lock = threading.RLock()
estado = {
    "deepseek_ativo": True,
    "gemini_ativo": bool(GEMINI_KEY),
    "ok": 0, "falhas": 0, "processados": 0, "inicio": time.time(),
}

thread_local = threading.local()


def get_session():
    if not hasattr(thread_local, "session"):
        s = requests.Session()
        s.headers.update({"User-Agent": "RAGI-2026-WordNet-OOInfo/1.0"})
        thread_local.session = s
    return thread_local.session


def agora():
    return datetime.now().isoformat(timespec="seconds")


def sleep_backoff(tentativa):
    espera = min(BACKOFF_MAX, BACKOFF_BASE ** tentativa) + random.uniform(0, 1)
    time.sleep(espera)


def slugify(texto):
    texto = str(texto).strip().replace("/", "_").replace("\\", "_")
    texto = "".join(c for c in texto if c.isalnum() or c in (" ", "_", "-"))
    return "_".join(texto.split())[:70] or "conceito"


# ================================================================
# 7. PROMPT VISUAL (DeepSeek -> Gemini -> template)
# ================================================================

SYSTEM_PROMPT = """You are an expert semantic visual designer creating images for an
educational lexical knowledge base.

Rules:
1. Decide whether the concept is CONCRETE or ABSTRACT.
2. CONCRETE: represent the meaning literally (the actual object, organism,
   place, event or scene).
3. ABSTRACT: do NOT depict the abstract word itself. Choose ONE strong
   concrete visual metaphor or scene that communicates the definition --
   prefer a widely recognizable symbol when one exists, otherwise invent a
   clear metaphor tied to the specific definition given.
4. Use the definition as the semantic authority.
5. The final scene must be drawable and visually understandable.
6. No text, no letters, no labels, no captions, no logos, no typography in
   the image. Do not put the lexical word in the image.
7. Clean educational editorial illustration style, square composition,
   neutral uncluttered background.

Return ONLY JSON: {"classificacao": "concreto"|"abstrato", "ideia_visual":
"short explanation", "prompt_visual": "final image prompt in English"}"""


def montar_contexto(item):
    partes = [f"Term: {item['termo']}", f"Definition in English: {item['descricao_en']}"]
    if item.get("descricao_pt"):
        partes.append("Definition in Portuguese: " + item["descricao_pt"])
    if item.get("classe_gramatical"):
        partes.append("Grammatical class: " + item["classe_gramatical"])
    partes.append("Return JSON only.")
    return "\n".join(partes)


def _extrair_json_planejamento(texto, modelo):
    texto = texto.strip()
    if texto.startswith("```"):
        texto = texto.replace("```json", "").replace("```", "").strip()
    resultado = json.loads(texto)
    prompt_visual = resultado.get("prompt_visual")
    if not prompt_visual:
        return None
    return {
        "classificacao": resultado.get("classificacao", ""),
        "ideia_visual": resultado.get("ideia_visual", ""),
        "prompt_visual": prompt_visual.strip(),
        "modelo": modelo,
    }


def gerar_prompt_deepseek(item, session):
    if not estado["deepseek_ativo"]:
        return None
    payload = {
        "model": DEEPSEEK_MODELO,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": montar_contexto(item)},
        ],
        "thinking": {"type": "disabled"},
        "response_format": {"type": "json_object"},
        "max_tokens": 500,
    }
    for tentativa in range(1, MAX_TENTATIVAS_DEEPSEEK + 1):
        try:
            resp = session.post(
                DEEPSEEK_URL,
                headers={"Authorization": f"Bearer {DEEPSEEK_KEY}", "Content-Type": "application/json"},
                json=payload, timeout=TIMEOUT_DEEPSEEK,
            )
            if resp.status_code == 200:
                texto = resp.json()["choices"][0]["message"]["content"]
                return _extrair_json_planejamento(texto, DEEPSEEK_MODELO)
            if resp.status_code == 429:
                with lock:
                    estado["deepseek_ativo"] = False
                print("⚠️ DeepSeek: rate limit/quota — desativada pro resto da execução.")
                return None
            if resp.status_code in (401, 403):
                with lock:
                    estado["deepseek_ativo"] = False
                print("⚠️ DeepSeek: chave rejeitada — desativada.")
                return None
            if resp.status_code < 500:
                return None
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError, json.JSONDecodeError, KeyError):
            pass
        if tentativa < MAX_TENTATIVAS_DEEPSEEK:
            sleep_backoff(tentativa)
    return None


def gerar_prompt_gemini(item, session):
    if not estado["gemini_ativo"]:
        return None
    prompt = f"{SYSTEM_PROMPT}\n\nNow process this concept:\n\n{montar_contexto(item)}"
    for modelo in GEMINI_MODELOS:
        for tentativa in range(1, MAX_TENTATIVAS_GEMINI + 1):
            try:
                url = f"https://generativelanguage.googleapis.com/v1beta/models/{modelo}:generateContent?key={GEMINI_KEY}"
                payload = {"contents": [{"parts": [{"text": prompt}]}],
                           "generationConfig": {"temperature": 0.2, "maxOutputTokens": 500}}
                resp = session.post(url, json=payload, timeout=TIMEOUT_GEMINI)
                if resp.status_code == 200:
                    texto = resp.json()["candidates"][0]["content"]["parts"][0]["text"]
                    resultado = _extrair_json_planejamento(texto, modelo)
                    if resultado:
                        return resultado
                    break
                if resp.status_code == 429:
                    with lock:
                        estado["gemini_ativo"] = False
                    print("⚠️ Gemini: limite/quota — desativado.")
                    return None
                if resp.status_code in (401, 403):
                    with lock:
                        estado["gemini_ativo"] = False
                    print("⚠️ Gemini: chave rejeitada — desativado.")
                    return None
                if resp.status_code == 404:
                    break  # tenta o próximo modelo da lista
            except (requests.exceptions.Timeout, requests.exceptions.ConnectionError, json.JSONDecodeError, KeyError):
                pass
            if tentativa < MAX_TENTATIVAS_GEMINI:
                sleep_backoff(tentativa)
    return None


def prompt_template(item):
    descricao = item.get("descricao_pt") or item.get("descricao_en") or item["termo"]
    return {
        "classificacao": "desconhecido",
        "ideia_visual": "Literal representation based on the dictionary definition.",
        "prompt_visual": (
            f"A clean educational editorial illustration depicting the following concept: {descricao}. "
            "Clear central subject, simple neutral background, square composition, soft lighting, "
            "no text, no letters, no labels, no logos."
        ),
        "modelo": "template",
    }


def criar_planejamento(item, session):
    resultado = gerar_prompt_deepseek(item, session)
    if resultado:
        return resultado
    resultado = gerar_prompt_gemini(item, session)
    if resultado:
        return resultado
    return prompt_template(item)


# ================================================================
# 8. POLLINATIONS (domínio legado confirmado funcionando)
# ================================================================

def gerar_imagem_replicate(prompt, seed):
    import replicate
    client_replicate = replicate.Client(api_token=REPLICATE_KEY)
    output = client_replicate.run(
        "black-forest-labs/flux-schnell",
        input={"prompt": prompt, "seed": seed},
    )
    arquivo = output[0] if isinstance(output, list) else output
    conteudo = arquivo.read()
    return conteudo, detectar_mime(conteudo)


def gerar_imagem(prompt, session, seed):
    """
    Pollinations legado como fonte principal.
    Replicate somente se Pollinations falhar completamente
    e houver chave configurada.
    """

    url = POLLINATIONS_URL + urllib.parse.quote(prompt, safe="")

    params = {
        "model": POLLINATIONS_MODELO,
        "width": LARGURA_IMAGEM,
        "height": ALTURA_IMAGEM,
        "seed": seed,
        "nologo": "true",
        "private": "true",
    }

    headers = {
        "Authorization": f"Bearer {POLLINATIONS_KEY}"
    }

    ultimo_erro = None

    for tentativa in range(1, MAX_TENTATIVAS_POLLINATIONS + 1):

        _aguardar_vez_pollinations()

        try:
            resp = session.get(
                url,
                params=params,
                headers=headers,
                timeout=TIMEOUT_POLLINATIONS,
            )

            if resp.status_code == 200:
                conteudo = resp.content

                if len(conteudo) < 10_000:
                    raise RuntimeError(
                        "Imagem recebida é suspeitamente pequena."
                    )

                return conteudo, detectar_mime(conteudo)

            if resp.status_code == 429:
                retry_after = resp.headers.get("Retry-After")

                if retry_after:
                    try:
                        espera = min(float(retry_after), 60)
                        time.sleep(espera)
                    except ValueError:
                        sleep_backoff(tentativa)
                else:
                    sleep_backoff(tentativa)

                ultimo_erro = "HTTP 429"
                continue

            if resp.status_code in (400, 401, 403, 404):
                raise RuntimeError(
                    f"Pollinations HTTP {resp.status_code}: "
                    f"{resp.text[:300]}"
                )

            ultimo_erro = f"HTTP {resp.status_code}"

        except (
            requests.exceptions.Timeout,
            requests.exceptions.ConnectionError,
        ) as e:
            ultimo_erro = str(e)

        except RuntimeError:
            raise

        if tentativa < MAX_TENTATIVAS_POLLINATIONS:
            sleep_backoff(tentativa)

    # ------------------------------------------------------------
    # Pollinations falhou completamente
    # ------------------------------------------------------------

    if REPLICATE_KEY:
        print(
            f"   ↪ Pollinations falhou ({str(ultimo_erro)[:100]}) "
            f"— tentando Replicate..."
        )
        return gerar_imagem_replicate(prompt, seed)

    raise RuntimeError(
        f"Pollinations falhou após {MAX_TENTATIVAS_POLLINATIONS} tentativas: "
        f"{ultimo_erro}"
    )

# ================================================================
# 9. UPLOAD OOINFO (base64, formato confirmado)
# ================================================================

def enviar_imagem_ooinfo(session, token, item_id, conteudo, mime):
    data_uri = f"data:{mime};base64," + base64.b64encode(conteudo).decode("ascii")
    return session.patch(
        f"{BASE_URL}/api/lists/{ID_LISTA_CONCEITOS}/items/{item_id}",
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
        json={"values": {CAMPO_IMAGEM: data_uri}}, timeout=TIMEOUT_OOINFO,
    )


# ================================================================
# 10. PROCESSAMENTO DE 1 ITEM (3 estágios com checkpoint)
# ================================================================

def processar_item(item):
    nosso_id = item["id"]
    session = get_session()

    with lock:
        registro = dict(progresso.get(nosso_id, {}))

    nome_imagem = slugify(item["termo"]) + "_" + hashlib.sha1(nosso_id.encode()).hexdigest()[:10] + ".png"
    caminho_imagem = PASTA_IMAGENS / nome_imagem

    # já estava tudo pronto de uma rodada anterior?
    if registro.get("status") == "ok" and imagem_valida(caminho_imagem):
        return {"id": nosso_id, "termo": item["termo"], "status": "ok", "reutilizado": True}

    # --- estágio 1: prompt ---
    planejamento = registro.get("planejamento")
    if not planejamento:
        planejamento = criar_planejamento(item, session)
        with lock:
            registro["planejamento"] = planejamento
            registro["status"] = "prompt_ok"
            registro["timestamp_prompt"] = agora()
            progresso[nosso_id] = registro
    prompt = planejamento["prompt_visual"]

    # --- estágio 2: imagem (reaproveita do disco se já existir) ---
    if imagem_valida(caminho_imagem):
        with open(caminho_imagem, "rb") as f:
            conteudo = f.read()
        mime = detectar_mime(conteudo)
    else:
        seed = int(hashlib.md5(nosso_id.encode()).hexdigest()[:8], 16) % 2_147_483_647
        conteudo, mime = gerar_imagem(prompt, session, seed)
        with open(caminho_imagem, "wb") as f:
            f.write(conteudo)
        with lock:
            registro["imagem_path"] = str(caminho_imagem)
            registro["seed"] = seed
            registro["status"] = "imagem_ok"
            registro["timestamp_imagem"] = agora()
            progresso[nosso_id] = registro

    # --- estágio 3: upload pro ooinfo ---
    for tentativa in range(1, MAX_TENTATIVAS_OOINFO + 1):
        resp = enviar_imagem_ooinfo(session, obter_token(), item["id_interno"], conteudo, mime)

        if resp.status_code in (200, 201):
            with lock:
                registro["status"] = "ok"
                registro["prompt"] = prompt
                registro["origem_prompt"] = planejamento.get("modelo", "desconhecido")
                registro["timestamp"] = agora()
                progresso[nosso_id] = registro
            return {"id": nosso_id, "termo": item["termo"], "status": "ok", "reutilizado": False}

        if resp.status_code == 401:
            renovar_token()
            continue
        if resp.status_code in (429, 500, 502, 503, 504) and tentativa < MAX_TENTATIVAS_OOINFO:
            sleep_backoff(tentativa)
            continue

        raise RuntimeError(f"ooinfo HTTP {resp.status_code}: {resp.text[:500]}")

    raise RuntimeError("ooinfo falhou após retries.")


# ================================================================
# 11. LISTA DE PENDENTES E EXECUÇÃO PARALELA
# ================================================================

pendentes = []
ja_ok = 0
for item in itens_trabalho:
    registro = progresso.get(item["id"], {})
    if registro.get("status") == "ok" and imagem_valida(registro.get("imagem_path", "")):
        ja_ok += 1
    else:
        pendentes.append(item)

print("\n" + "=" * 70)
print("INICIANDO PRODUÇÃO")
print("=" * 70)
print(f"Total no escopo: {len(itens_trabalho):,} | Já concluídos: {ja_ok:,} | Pendentes: {len(pendentes):,}")
print(f"Workers: {NUM_WORKERS}")
print("=" * 70)

inicio_rodada = time.time()
estado["inicio"] = inicio_rodada


def imprimir_progresso():
    with lock:
        estado["processados"] += 1
        decorrido = time.time() - estado["inicio"]
        ritmo = estado["processados"] / decorrido if decorrido > 0 else 0
        restantes = max(0, len(pendentes) - estado["processados"])
        eta = restantes / ritmo if ritmo > 0 else 0
        print(f"   [{estado['processados']:,}/{len(pendentes):,}] "
              f"OK: {estado['ok']:,} | Falhas: {estado['falhas']:,} | "
              f"ritmo: {ritmo*60:.0f}/min | ETA: {eta/60:.1f} min")
        if estado["processados"] % SALVAR_A_CADA == 0:
            salvar_checkpoint()


# ================================================================
# EXECUÇÃO CONTROLADA
# ================================================================

TAMANHO_LOTE = NUM_WORKERS * 4

total_pendentes = len(pendentes)

print(
    f"\nProcessamento controlado em lotes de "
    f"{TAMANHO_LOTE} itens."
)

for inicio_lote in range(0, total_pendentes, TAMANHO_LOTE):

    lote = pendentes[
        inicio_lote:min(inicio_lote + TAMANHO_LOTE, total_pendentes)
    ]

    print(
        f"\nLote "
        f"{inicio_lote + 1:,}–"
        f"{min(inicio_lote + len(lote), total_pendentes):,} "
        f"de {total_pendentes:,}"
    )

    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:

        futuros = {
            executor.submit(processar_item, item): item
            for item in lote
        }

        for futuro in as_completed(futuros):

            item = futuros[futuro]

            try:
                futuro.result()

                with lock:
                    estado["ok"] += 1

            except Exception as e:

                with lock:
                    estado["falhas"] += 1

                    progresso[item["id"]] = {
                        **progresso.get(item["id"], {}),
                        "status": "falha",
                        "erro": str(e),
                        "timestamp": agora(),
                    }

                print(
                    f"✗ {item['termo']} | "
                    f"{str(e)[:180]}"
                )

            imprimir_progresso()

    # checkpoint ao terminar cada lote
    with lock:
        salvar_checkpoint()

    print(
        f"✓ Lote concluído | "
        f"Checkpoint salvo | "
        f"Progresso: "
        f"{min(inicio_lote + len(lote), total_pendentes):,}/"
        f"{total_pendentes:,}"
    )

with lock:
    salvar_checkpoint()

# ================================================================
# 12. RESUMO FINAL
# ================================================================

tempo_total = time.time() - inicio_rodada
total_ok = sum(1 for r in progresso.values() if r.get("status") == "ok")
total_falha = sum(1 for r in progresso.values() if r.get("status") == "falha")

print("\n\n" + "=" * 70)
print("PRODUÇÃO ENCERRADA")
print("=" * 70)
print(f"Processados nesta rodada: {estado['processados']:,}")
print(f"Sucesso nesta rodada: {estado['ok']:,} | Falhas nesta rodada: {estado['falhas']:,}")
print(f"Total geral OK: {total_ok:,}/{len(itens_trabalho):,} | Total com falha: {total_falha:,}")
print(f"Tempo desta rodada: {tempo_total/60:.1f} min")
if tempo_total > 0:
    print(f"Velocidade média: {estado['processados']/tempo_total*60:.0f} itens/min")
print(f"\nCheckpoint: {ARQUIVO_PROGRESSO}")
print(f"Pasta das imagens: {PASTA_IMAGENS}")
print("=" * 70)


Drive not mounted, so nothing to flush and unmount.
Google Drive desmontado com sucesso.
Limpando o diretório /content/drive...
Removido: /content/drive/MyDrive
Diretório /content/drive limpo.
Mounted at /content/drive
BASE CARREGADA
Conceitos: 43,895 | Traduções PT: 39,782

Escopo (FILTRO_SUBSTANTIVO=False): 43,895
Prontos para produção: 43,895
Sem mapeamento ooinfo: 0 | Sem termo encontrado: 0

AUDITORIA DO CHECKPOINT
Registros no checkpoint: 1,707
  falha: 790
  ok: 917
Registros com caminho de imagem: 790
Imagens válidas encontradas: 790

📧 Email do ooinfo: eduardoparente@discente.ufg.br


KeyboardInterrupt: Interrupted by user

In [ ]:
import os
import json
import time
import random
import base64
import hashlib
import threading
import urllib.parse
import requests
import shutil

from pathlib import Path
from datetime import datetime
from getpass import getpass
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.colab import drive

# ================================================================
# 1. CONFIGURAÇÕES
# ================================================================

RAIZ_PROJETO = Path("/content/drive/MyDrive/Grupo 3 - RAGI 2026")
PASTA_WORDNET = RAIZ_PROJETO / "Datasets" / "Wordnet"
PASTA_IMAGENS = RAIZ_PROJETO / "Imagens" / "Conceitos"
PASTA_IMAGENS.mkdir(parents=True, exist_ok=True)

ARQUIVO_CONCEITOS = PASTA_WORDNET / "conceitos.json"
ARQUIVO_PALAVRAS = PASTA_WORDNET / "palavras.json"
ARQUIVO_MAPEAMENTO = PASTA_WORDNET / "mapeamento_conceitos_ooinfo.json"
ARQUIVO_TRADUCOES = RAIZ_PROJETO / "Traducao" / "conceitos_traduzidos.json"
ARQUIVO_PROGRESSO = PASTA_WORDNET / "progresso_imagens_conceitos.json"

BASE_URL = "https://ooinfo.org"
ID_LISTA_CONCEITOS = "cmsyyeh9d001b35btu2ginbbv"
CAMPO_IMAGEM = "imagem"

DEEPSEEK_URL = "https://api.deepseek.com/chat/completions"
DEEPSEEK_MODELO = "deepseek-v4-flash"

GEMINI_MODELOS = ["gemini-3.7-flash", "gemini-2.5-flash-lite"]

# Domínio LEGADO confirmado funcionando (o moderno devolveu 401 pra essa chave)
POLLINATIONS_URL = "https://image.pollinations.ai/prompt/"
POLLINATIONS_MODELO = "flux"
LARGURA_IMAGEM = 512
ALTURA_IMAGEM = 512

FILTRO_SUBSTANTIVO = False  # True = só Substantivo (~32.674). False = todos (~43.895).

NUM_WORKERS = 2
SALVAR_A_CADA = 50
MAX_TENTATIVAS_DEEPSEEK = 3
MAX_TENTATIVAS_GEMINI = 2
MAX_TENTATIVAS_POLLINATIONS = 12
MAX_TENTATIVAS_OOINFO = 3
TIMEOUT_DEEPSEEK = 45
TIMEOUT_GEMINI = 45
TIMEOUT_POLLINATIONS = (15, 120)
TIMEOUT_OOINFO = 60
BACKOFF_BASE = 2
BACKOFF_MAX = 30
POLLINATIONS_MIN_INTERVAL = 15.0

# Limitador de taxa dedicado ao Pollinations: no máximo 1 requisição de cada
# vez, com esse intervalo mínimo entre chamadas -- independente de quantos
# workers estejam rodando ao mesmo tempo. Comece conservador; depois de rodar
# limpo por uns minutos sem 429, reduza aos poucos (ex: 2.0). Vale conferir
# também o dashboard em enter.pollinations.ai pra ver o limite real da chave.
POLLINATIONS_MIN_INTERVAL = 5.0
_pollinations_lock = threading.Lock()
_pollinations_proximo_horario = [0.0]


def _aguardar_vez_pollinations():
    with _pollinations_lock:
        agora = time.time()

        espera = _pollinations_proximo_horario[0] - agora

        if espera > 0:
            time.sleep(espera)

        # IMPORTANTE:
        # recalcula o horário depois do sleep
        agora = time.time()

        _pollinations_proximo_horario[0] = (
            agora + POLLINATIONS_MIN_INTERVAL
        )

# ================================================================
# 2. DRIVE E ARQUIVOS
# ================================================================

# Desmonta e limpa o ponto de montagem do Drive para evitar `ValueError: Mountpoint must not already contain files`
if os.path.exists('/content/drive'):
    try:
        drive.flush_and_unmount()
        print("Google Drive desmontado com sucesso.")
    except ValueError:
        print("Google Drive não estava montado ou já foi desmontado.")
    except Exception as e:
        print(f"Erro ao tentar desmontar Google Drive: {e}")

# Garante que o diretório '/content/drive' esteja vazio.
if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    print("Limpando o diretório /content/drive...")
    for item in os.listdir('/content/drive'):
        item_path = os.path.join('/content/drive', item)
        try:
            if os.path.isfile(item_path) or os.path.islink(item_path):
                os.remove(item_path)
            elif os.path.isdir(item_path):
                shutil.rmtree(item_path)
            print(f"Removido: {item_path}")
        except Exception as e:
            print(f"Erro ao remover {item_path}: {e}")
    print("Diretório /content/drive limpo.")

drive.mount("/content/drive", force_remount=False)

for arquivo in (ARQUIVO_CONCEITOS, ARQUIVO_PALAVRAS, ARQUIVO_MAPEAMENTO):
    if not arquivo.exists():
        raise FileNotFoundError(f"Arquivo não encontrado:\n{arquivo}")

with open(ARQUIVO_CONCEITOS, "r", encoding="utf-8") as f:
    conceitos = json.load(f)
with open(ARQUIVO_PALAVRAS, "r", encoding="utf-8") as f:
    palavras = json.load(f)
with open(ARQUIVO_MAPEAMENTO, "r", encoding="utf-8") as f:
    mapeamento_conceitos = json.load(f)

traducoes_por_id = {}
if ARQUIVO_TRADUCOES.exists():
    with open(ARQUIVO_TRADUCOES, "r", encoding="utf-8") as f:
        traducoes = json.load(f)
    traducoes_por_id = {t["id"]: t.get("definicao_pt") for t in traducoes if t.get("definicao_pt")}

palavra_por_id = {p["id"]: p.get("palavra") or p.get("term") or p.get("label") or "" for p in palavras}

print("=" * 70)
print("BASE CARREGADA")
print("=" * 70)
print(f"Conceitos: {len(conceitos):,} | Traduções PT: {len(traducoes_por_id):,}")

# ================================================================
# 3. ESCOPO DE TRABALHO (com filtro por Substantivo)
# ================================================================

escopo = conceitos
if FILTRO_SUBSTANTIVO:
    escopo = [c for c in conceitos if c.get("classe_gramatical") == "Substantivo"]

itens_trabalho = []
sem_mapeamento = 0
sem_termo = 0

for c in escopo:
    nosso_id = c.get("id")
    if not nosso_id:
        continue
    id_interno = mapeamento_conceitos.get(nosso_id)
    if not id_interno:
        sem_mapeamento += 1
        continue

    termo = palavra_por_id.get(c.get("termo_preferencial"))
    if not termo:
        for pid in c.get("palavras") or []:
            if isinstance(pid, dict):
                pid = pid.get("id")
            candidato = palavra_por_id.get(pid)
            if candidato:
                termo = candidato
                break
    if not termo:
        termo = nosso_id
        sem_termo += 1

    itens_trabalho.append({
        "id": nosso_id,
        "id_interno": id_interno,
        "termo": termo,
        "descricao_en": (c.get("definicao") or "").strip(),
        "descricao_pt": (traducoes_por_id.get(nosso_id) or "").strip(),
        "classe_gramatical": c.get("classe_gramatical") or "",
    })

print(f"\nEscopo (FILTRO_SUBSTANTIVO={FILTRO_SUBSTANTIVO}): {len(escopo):,}")
print(f"Prontos para produção: {len(itens_trabalho):,}")
print(f"Sem mapeamento ooinfo: {sem_mapeamento:,} | Sem termo encontrado: {sem_termo:,}")

# ================================================================
# 4. CHECKPOINT
# ================================================================

def carregar_checkpoint():
    if not ARQUIVO_PROGRESSO.exists():
        return {}
    try:
        with open(ARQUIVO_PROGRESSO, "r", encoding="utf-8") as f:
            dados = json.load(f)
        return dados if isinstance(dados, dict) else {}
    except Exception as e:
        print(f"⚠️ Erro no checkpoint: {e}")
        return {}


def salvar_checkpoint():
    temporario = str(ARQUIVO_PROGRESSO) + ".tmp"
    with open(temporario, "w", encoding="utf-8") as f:
        json.dump(progresso, f, ensure_ascii=False, indent=2)
    os.replace(temporario, ARQUIVO_PROGRESSO)


def detectar_mime(conteudo):
    if conteudo[:3] == b"\xff\xd8\xff":
        return "image/jpeg"
    if conteudo[:8] == b"\x89PNG\r\n\x1a\n":
        return "image/png"
    if conteudo[:4] == b"RIFF":
        return "image/webp"
    return "image/png"


def imagem_valida(caminho):
    if not caminho:
        return False

    try:
        caminho = Path(caminho)
        return (
            caminho.exists()
            and caminho.is_file()
            and caminho.stat().st_size >= 10_000
        )
    except (TypeError, OSError):
        return False


progresso = carregar_checkpoint()

print("\n" + "=" * 70)
print("AUDITORIA DO CHECKPOINT")
print("=" * 70)

status_count = {}

for registro in progresso.values():
    status = registro.get("status", "sem_status")
    status_count[status] = status_count.get(status, 0) + 1

print(f"Registros no checkpoint: {len(progresso):,}")

for status, quantidade in sorted(status_count.items()):
    print(f"  {status}: {quantidade:,}")

imagens_checkpoint = 0
imagens_validas = 0

for registro in progresso.values():

    caminho = registro.get("imagem_path")

    if caminho:
        imagens_checkpoint += 1

        if imagem_valida(caminho):
            imagens_validas += 1

print(f"Registros com caminho de imagem: {imagens_checkpoint:,}")
print(f"Imagens válidas encontradas: {imagens_validas:,}")

print("=" * 70)

# ================================================================
# 5. LOGIN OOINFO E CHAVES
# ================================================================

EMAIL_OOINFO = input("\n📧 Email do ooinfo: ")
SENHA_OOINFO = getpass("🔐 Senha do ooinfo: ")


def fazer_login():
    resp = requests.post(
        f"{BASE_URL}/api/auth/login",
        json={"email": EMAIL_OOINFO, "password": SENHA_OOINFO}, timeout=30,
    )
    if resp.status_code not in (200, 201):
        raise RuntimeError("Falha no login ooinfo: " + resp.text[:500])
    token = resp.json().get("accessToken")
    if not token:
        raise RuntimeError("ooinfo não retornou accessToken.")
    return token


print("\n" + "=" * 70)
print("AUTENTICAÇÃO")
print("=" * 70)
TOKEN_OOINFO = fazer_login()
print("✓ Login ooinfo OK.")

DEEPSEEK_KEY = getpass("🔑 DeepSeek API Key: ")
GEMINI_KEY = getpass("🔑 Gemini API Key (fallback, Enter p/ pular): ")
POLLINATIONS_KEY = getpass("🔑 Pollinations API Key: ")
REPLICATE_KEY = getpass("🔑 Replicate API Key (fallback pago, ~US$0.003/img — Enter p/ pular): ")

token_lock = threading.Lock()


def obter_token():
    return TOKEN_OOINFO


def renovar_token():
    global TOKEN_OOINFO
    with token_lock:
        TOKEN_OOINFO = fazer_login()
        return TOKEN_OOINFO


# ================================================================
# 6. ESTADO GLOBAL, SESSÃO POR THREAD, UTILITÁRIOS
# ================================================================

lock = threading.RLock()
estado = {
    "deepseek_ativo": True,
    "gemini_ativo": bool(GEMINI_KEY),
    "ok": 0, "falhas": 0, "processados": 0, "inicio": time.time(),
}

thread_local = threading.local()


def get_session():
    if not hasattr(thread_local, "session"):
        s = requests.Session()
        s.headers.update({"User-Agent": "RAGI-2026-WordNet-OOInfo/1.0"})
        thread_local.session = s
    return thread_local.session


def agora():
    return datetime.now().isoformat(timespec="seconds")


def sleep_backoff(tentativa):
    espera = min(BACKOFF_MAX, BACKOFF_BASE ** tentativa) + random.uniform(0, 1)
    time.sleep(espera)


def slugify(texto):
    texto = str(texto).strip().replace("/", "_").replace("\\", "_")
    texto = "".join(c for c in texto if c.isalnum() or c in (" ", "_", "-"))
    return "_".join(texto.split())[:70] or "conceito"


# ================================================================
# 7. PROMPT VISUAL (DeepSeek -> Gemini -> template)
# ================================================================

SYSTEM_PROMPT = """You are an expert semantic visual designer creating images for an
educational lexical knowledge base.

Rules:
1. Decide whether the concept is CONCRETE or ABSTRACT.
2. CONCRETE: represent the meaning literally (the actual object, organism,
   place, event or scene).
3. ABSTRACT: do NOT depict the abstract word itself. Choose ONE strong
   concrete visual metaphor or scene that communicates the definition --
   prefer a widely recognizable symbol when one exists, otherwise invent a
   clear metaphor tied to the specific definition given.
4. Use the definition as the semantic authority.
5. The final scene must be drawable and visually understandable.
6. No text, no letters, no labels, no captions, no logos, no typography in
   the image. Do not put the lexical word in the image.
7. Clean educational editorial illustration style, square composition,
   neutral uncluttered background.

Return ONLY JSON: {"classificacao": "concreto"|"abstrato", "ideia_visual":
"short explanation", "prompt_visual": "final image prompt in English"}"""


def montar_contexto(item):
    partes = [f"Term: {item['termo']}", f"Definition in English: {item['descricao_en']}"]
    if item.get("descricao_pt"):
        partes.append("Definition in Portuguese: " + item["descricao_pt"])
    if item.get("classe_gramatical"):
        partes.append("Grammatical class: " + item["classe_gramatical"])
    partes.append("Return JSON only.")
    return "\n".join(partes)


def _extrair_json_planejamento(texto, modelo):
    texto = texto.strip()
    if texto.startswith("```"):
        texto = texto.replace("```json", "").replace("```", "").strip()
    resultado = json.loads(texto)
    prompt_visual = resultado.get("prompt_visual")
    if not prompt_visual:
        return None
    return {
        "classificacao": resultado.get("classificacao", ""),
        "ideia_visual": resultado.get("ideia_visual", ""),
        "prompt_visual": prompt_visual.strip(),
        "modelo": modelo,
    }


def gerar_prompt_deepseek(item, session):
    if not estado["deepseek_ativo"]:
        return None
    payload = {
        "model": DEEPSEEK_MODELO,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": montar_contexto(item)},
        ],
        "thinking": {"type": "disabled"},
        "response_format": {"type": "json_object"},
        "max_tokens": 500,
    }
    for tentativa in range(1, MAX_TENTATIVAS_DEEPSEEK + 1):
        try:
            resp = session.post(
                DEEPSEEK_URL,
                headers={"Authorization": f"Bearer {DEEPSEEK_KEY}", "Content-Type": "application/json"},
                json=payload, timeout=TIMEOUT_DEEPSEEK,
            )
            if resp.status_code == 200:
                texto = resp.json()["choices"][0]["message"]["content"]
                return _extrair_json_planejamento(texto, DEEPSEEK_MODELO)
            if resp.status_code == 429:
                with lock:
                    estado["deepseek_ativo"] = False
                print("⚠️ DeepSeek: rate limit/quota — desativada pro resto da execução.")
                return None
            if resp.status_code in (401, 403):
                with lock:
                    estado["deepseek_ativo"] = False
                print("⚠️ DeepSeek: chave rejeitada — desativada.")
                return None
            if resp.status_code < 500:
                return None
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError, json.JSONDecodeError, KeyError):
            pass
        if tentativa < MAX_TENTATIVAS_DEEPSEEK:
            sleep_backoff(tentativa)
    return None


def gerar_prompt_gemini(item, session):
    if not estado["gemini_ativo"]:
        return None
    prompt = f"{SYSTEM_PROMPT}\n\nNow process this concept:\n\n{montar_contexto(item)}"
    for modelo in GEMINI_MODELOS:
        for tentativa in range(1, MAX_TENTATIVAS_GEMINI + 1):
            try:
                url = f"https://generativelanguage.googleapis.com/v1beta/models/{modelo}:generateContent?key={GEMINI_KEY}"
                payload = {"contents": [{"parts": [{"text": prompt}]}],
                           "generationConfig": {"temperature": 0.2, "maxOutputTokens": 500}}
                resp = session.post(url, json=payload, timeout=TIMEOUT_GEMINI)
                if resp.status_code == 200:
                    texto = resp.json()["candidates"][0]["content"]["parts"][0]["text"]
                    resultado = _extrair_json_planejamento(texto, modelo)
                    if resultado:
                        return resultado
                    break
                if resp.status_code == 429:
                    with lock:
                        estado["gemini_ativo"] = False
                    print("⚠️ Gemini: limite/quota — desativado.")
                    return None
                if resp.status_code in (401, 403):
                    with lock:
                        estado["gemini_ativo"] = False
                    print("⚠️ Gemini: chave rejeitada — desativado.")
                    return None
                if resp.status_code == 404:
                    break  # tenta o próximo modelo da lista
            except (requests.exceptions.Timeout, requests.exceptions.ConnectionError, json.JSONDecodeError, KeyError):
                pass
            if tentativa < MAX_TENTATIVAS_GEMINI:
                sleep_backoff(tentativa)
    return None


def prompt_template(item):
    descricao = item.get("descricao_pt") or item.get("descricao_en") or item["termo"]
    return {
        "classificacao": "desconhecido",
        "ideia_visual": "Literal representation based on the dictionary definition.",
        "prompt_visual": (
            f"A clean educational editorial illustration depicting the following concept: {descricao}. "
            "Clear central subject, simple neutral background, square composition, soft lighting, "
            "no text, no letters, no labels, no logos."
        ),
        "modelo": "template",
    }


def criar_planejamento(item, session):
    resultado = gerar_prompt_deepseek(item, session)
    if resultado:
        return resultado
    resultado = gerar_prompt_gemini(item, session)
    if resultado:
        return resultado
    return prompt_template(item)


# ================================================================
# 8. POLLINATIONS (domínio legado confirmado funcionando)
# ================================================================

def gerar_imagem(prompt, session, seed):
    url = POLLINATIONS_URL + urllib.parse.quote(prompt, safe="")

    params = {
        "model": POLLINATIONS_MODELO,
        "width": LARGURA_IMAGEM,
        "height": ALTURA_IMAGEM,
        "seed": seed,
        "nologo": "true",
        "private": "true",
    }

    headers = {
        "Authorization": f"Bearer {POLLINATIONS_KEY}"
    }

    ultimo_erro = None

    for tentativa in range(1, MAX_TENTATIVAS_POLLINATIONS + 1):

        _aguardar_vez_pollinations()

        try:
            resp = session.get(
                url,
                params=params,
                headers=headers,
                timeout=TIMEOUT_POLLINATIONS
            )

            # ==========================================
            # SUCESSO
            # ==========================================
            if resp.status_code == 200:

                conteudo = resp.content

                if len(conteudo) < 10_000:
                    ultimo_erro = "Imagem recebida é suspeitamente pequena."
                else:
                    print(
                        f"✓ Pollinations OK "
                        f"(tentativa {tentativa}/{MAX_TENTATIVAS_POLLINATIONS})"
                    )

                    return conteudo, detectar_mime(conteudo)

            # ==========================================
            # RATE LIMIT
            # ==========================================
            elif resp.status_code == 429:

                retry_after = resp.headers.get("Retry-After")

                if retry_after:
                    try:
                        espera = float(retry_after)
                    except ValueError:
                        espera = BACKOFF_BASE ** tentativa
                else:
                    espera = BACKOFF_BASE ** tentativa

                espera = min(espera, BACKOFF_MAX)

                ultimo_erro = "HTTP 429"

                print(
                    f"⏳ Pollinations 429 "
                    f"(tentativa {tentativa}/{MAX_TENTATIVAS_POLLINATIONS}) "
                    f"— aguardando {espera:.1f}s"
                )

                time.sleep(espera)

                continue

            # ==========================================
            # ERROS TRANSITÓRIOS
            # ==========================================
            elif resp.status_code in (500, 502, 503, 504):

                ultimo_erro = f"HTTP {resp.status_code}"

                espera = min(
                    BACKOFF_BASE ** tentativa,
                    BACKOFF_MAX
                )

                print(
                    f"⚠ Pollinations {ultimo_erro} "
                    f"(tentativa {tentativa}/{MAX_TENTATIVAS_POLLINATIONS}) "
                    f"— aguardando {espera}s"
                )

                time.sleep(espera)

                continue

            # ==========================================
            # ERROS DEFINITIVOS
            # ==========================================
            elif resp.status_code in (400, 401, 403, 404):

                raise RuntimeError(
                    f"Pollinations HTTP {resp.status_code}: "
                    f"{resp.text[:300]}"
                )

            else:

                ultimo_erro = f"HTTP {resp.status_code}"

                espera = min(
                    BACKOFF_BASE ** tentativa,
                    BACKOFF_MAX
                )

                time.sleep(espera)

        except (
            requests.exceptions.Timeout,
            requests.exceptions.ConnectionError
        ) as e:

            ultimo_erro = str(e)

            espera = min(
                BACKOFF_BASE ** tentativa,
                BACKOFF_MAX
            )

            print(
                f"⚠ Erro de conexão Pollinations "
                f"(tentativa {tentativa}/{MAX_TENTATIVAS_POLLINATIONS}) "
                f"— aguardando {espera}s"
            )

            time.sleep(espera)

    # ==========================================
    # FALLBACK REPLICATE
    # ==========================================
    if REPLICATE_KEY:
        print(
            "↪ Pollinations falhou após todas as tentativas "
            "— tentando Replicate..."
        )

        return gerar_imagem_replicate(prompt, seed)

    raise RuntimeError(
        f"Pollinations falhou após "
        f"{MAX_TENTATIVAS_POLLINATIONS} tentativas: "
        f"{ultimo_erro}"
    )

    # ------------------------------------------------------------
    # Pollinations falhou completamente
    # ------------------------------------------------------------

    if REPLICATE_KEY:
        print(
            f"   ↪ Pollinations falhou ({str(ultimo_erro)[:100]}) "
            f"— tentando Replicate..."
        )
        return gerar_imagem_replicate(prompt, seed)

    raise RuntimeError(
        f"Pollinations falhou após {MAX_TENTATIVAS_POLLINATIONS} tentativas: "
        f"{ultimo_erro}"
    )

# ================================================================
# 9. UPLOAD OOINFO (base64, formato confirmado)
# ================================================================

def enviar_imagem_ooinfo(session, token, item_id, conteudo, mime):
    data_uri = f"data:{mime};base64," + base64.b64encode(conteudo).decode("ascii")
    return session.patch(
        f"{BASE_URL}/api/lists/{ID_LISTA_CONCEITOS}/items/{item_id}",
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
        json={"values": {CAMPO_IMAGEM: data_uri}}, timeout=TIMEOUT_OOINFO,
    )


# ================================================================
# 10. PROCESSAMENTO DE 1 ITEM (3 estágios com checkpoint)
# ================================================================

def processar_item(item):
    nosso_id = item["id"]
    session = get_session()

    with lock:
        registro = dict(progresso.get(nosso_id, {}))

    nome_imagem = slugify(item["termo"]) + "_" + hashlib.sha1(nosso_id.encode()).hexdigest()[:10] + ".png"
    caminho_imagem = PASTA_IMAGENS / nome_imagem

    # já estava tudo pronto de uma rodada anterior?
    if registro.get("status") == "ok" and imagem_valida(caminho_imagem):
        return {"id": nosso_id, "termo": item["termo"], "status": "ok", "reutilizado": True}

    # --- estágio 1: prompt ---
    planejamento = registro.get("planejamento")
    if not planejamento:
        planejamento = criar_planejamento(item, session)
        with lock:
            registro["planejamento"] = planejamento
            registro["status"] = "prompt_ok"
            registro["timestamp_prompt"] = agora()
            progresso[nosso_id] = registro
    prompt = planejamento["prompt_visual"]

    # --- estágio 2: imagem (reaproveita do disco se já existir) ---
    if imagem_valida(caminho_imagem):
        with open(caminho_imagem, "rb") as f:
            conteudo = f.read()
        mime = detectar_mime(conteudo)
    else:
        seed = int(hashlib.md5(nosso_id.encode()).hexdigest()[:8], 16) % 2_147_483_647
        conteudo, mime = gerar_imagem(prompt, session, seed)
        with open(caminho_imagem, "wb") as f:
            f.write(conteudo)
        with lock:
            registro["imagem_path"] = str(caminho_imagem)
            registro["seed"] = seed
            registro["status"] = "imagem_ok"
            registro["timestamp_imagem"] = agora()
            progresso[nosso_id] = registro

    # --- estágio 3: upload pro ooinfo ---
    for tentativa in range(1, MAX_TENTATIVAS_OOINFO + 1):
        resp = enviar_imagem_ooinfo(session, obter_token(), item["id_interno"], conteudo, mime)

        if resp.status_code in (200, 201):
            with lock:
                registro["status"] = "ok"
                registro["prompt"] = prompt
                registro["origem_prompt"] = planejamento.get("modelo", "desconhecido")
                registro["timestamp"] = agora()
                progresso[nosso_id] = registro
            return {"id": nosso_id, "termo": item["termo"], "status": "ok", "reutilizado": False}

        if resp.status_code == 401:
            renovar_token()
            continue
        if resp.status_code in (429, 500, 502, 503, 504) and tentativa < MAX_TENTATIVAS_OOINFO:
            sleep_backoff(tentativa)
            continue

        raise RuntimeError(f"ooinfo HTTP {resp.status_code}: {resp.text[:500]}")

    raise RuntimeError("ooinfo falhou após retries.")


# ================================================================
# 11. LISTA DE PENDENTES E EXECUÇÃO PARALELA
# ================================================================

pendentes = []
ja_ok = 0
for item in itens_trabalho:
    registro = progresso.get(item["id"], {})
    if registro.get("status") == "ok" and imagem_valida(registro.get("imagem_path", "")):
        ja_ok += 1
    else:
        pendentes.append(item)

print("\n" + "=" * 70)
print("INICIANDO PRODUÇÃO")
print("=" * 70)
print(f"Total no escopo: {len(itens_trabalho):,} | Já concluídos: {ja_ok:,} | Pendentes: {len(pendentes):,}")
print(f"Workers: {NUM_WORKERS}")
print("=" * 70)

inicio_rodada = time.time()
estado["inicio"] = inicio_rodada


def imprimir_progresso():
    with lock:
        estado["processados"] += 1
        decorrido = time.time() - estado["inicio"]
        ritmo = estado["processados"] / decorrido if decorrido > 0 else 0
        restantes = max(0, len(pendentes) - estado["processados"])
        eta = restantes / ritmo if ritmo > 0 else 0
        print(f"   [{estado['processados']:,}/{len(pendentes):,}] "
              f"OK: {estado['ok']:,} | Falhas: {estado['falhas']:,} | "
              f"ritmo: {ritmo*60:.0f}/min | ETA: {eta/60:.1f} min")
        if estado["processados"] % SALVAR_A_CADA == 0:
            salvar_checkpoint()


# ================================================================
# EXECUÇÃO CONTROLADA
# ================================================================

TAMANHO_LOTE = NUM_WORKERS * 4

total_pendentes = len(pendentes)

print(
    f"\nProcessamento controlado em lotes de "
    f"{TAMANHO_LOTE} itens."
)

for inicio_lote in range(0, total_pendentes, TAMANHO_LOTE):

    lote = pendentes[
        inicio_lote:min(inicio_lote + TAMANHO_LOTE, total_pendentes)
    ]

    print(
        f"\nLote "
        f"{inicio_lote + 1:,}–"
        f"{min(inicio_lote + len(lote), total_pendentes):,} "
        f"de {total_pendentes:,}"
    )

    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:

        futuros = {
            executor.submit(processar_item, item): item
            for item in lote
        }

        for futuro in as_completed(futuros):

            item = futuros[futuro]

            try:
                futuro.result()

                with lock:
                    estado["ok"] += 1

            except Exception as e:

                with lock:
                    estado["falhas"] += 1

                    progresso[item["id"]] = {
                        **progresso.get(item["id"], {}),
                        "status": "falha",
                        "erro": str(e),
                        "timestamp": agora(),
                    }

                print(
                    f"✗ {item['termo']} | "
                    f"{str(e)[:180]}"
                )

            imprimir_progresso()

    # checkpoint ao terminar cada lote
    with lock:
        salvar_checkpoint()

    print(
        f"✓ Lote concluído | "
        f"Checkpoint salvo | "
        f"Progresso: "
        f"{min(inicio_lote + len(lote), total_pendentes):,}/"
        f"{total_pendentes:,}"
    )

with lock:
    salvar_checkpoint()

# ================================================================
# 12. RESUMO FINAL
# ================================================================

tempo_total = time.time() - inicio_rodada
total_ok = sum(1 for r in progresso.values() if r.get("status") == "ok")
total_falha = sum(1 for r in progresso.values() if r.get("status") == "falha")

print("\n\n" + "=" * 70)
print("PRODUÇÃO ENCERRADA")
print("=" * 70)
print(f"Processados nesta rodada: {estado['processados']:,}")
print(f"Sucesso nesta rodada: {estado['ok']:,} | Falhas nesta rodada: {estado['falhas']:,}")
print(f"Total geral OK: {total_ok:,}/{len(itens_trabalho):,} | Total com falha: {total_falha:,}")
print(f"Tempo desta rodada: {tempo_total/60:.1f} min")
if tempo_total > 0:
    print(f"Velocidade média: {estado['processados']/tempo_total*60:.0f} itens/min")
print(f"\nCheckpoint: {ARQUIVO_PROGRESSO}")
print(f"Pasta das imagens: {PASTA_IMAGENS}")
print("=" * 70)


Google Drive desmontado com sucesso.
Mounted at /content/drive
BASE CARREGADA
Conceitos: 43,895 | Traduções PT: 39,782

Escopo (FILTRO_SUBSTANTIVO=False): 43,895
Prontos para produção: 43,895
Sem mapeamento ooinfo: 0 | Sem termo encontrado: 0

AUDITORIA DO CHECKPOINT
Registros no checkpoint: 1,707
  falha: 790
  ok: 917
Registros com caminho de imagem: 790
Imagens válidas encontradas: 790

AUTENTICAÇÃO
✓ Login ooinfo OK.

INICIANDO PRODUÇÃO
Total no escopo: 43,895 | Já concluídos: 790 | Pendentes: 43,105
Workers: 2

Processamento controlado em lotes de 8 itens.

Lote 1–8 de 43,105
   [1/43,105] OK: 1 | Falhas: 0 | ritmo: 18851/min | ETA: 2.3 min
   [2/43,105] OK: 2 | Falhas: 0 | ritmo: 30791/min | ETA: 1.4 min
   [3/43,105] OK: 3 | Falhas: 0 | ritmo: 40597/min | ETA: 1.1 min
   [4/43,105] OK: 4 | Falhas: 0 | ritmo: 45700/min | ETA: 0.9 min
   [5/43,105] OK: 5 | Falhas: 0 | ritmo: 39180/min | ETA: 1.1 min
   [6/43,105] OK: 6 | Falhas: 0 | ritmo: 42410/min | ETA: 1.0 min
   [7/43,105] OK

KeyboardInterrupt: 

In [ ]:
# Recoloca todas as falhas na fila para nova tentativa
reprocessadas = 0

for conceito_id, registro in progresso.items():
    if registro.get("status") == "falha":
        registro["status"] = "pendente"
        registro.pop("erro", None)
        reprocessadas += 1

print(f"✓ {reprocessadas} falhas recolocadas na fila.")

✓ 937 falhas recolocadas na fila.
